In [ ]:

# ============================================================
# NOTEBOOK 10 — NSL-KDD RANDOM FOREST REPLICATION
# CELL 01 — ENVIRONMENT AND REPRODUCIBILITY VERIFICATION
# ============================================================

from google.colab import drive
from pathlib import Path
import hashlib
import json
import sys

import numpy as np
import pandas as pd
import pyarrow
import sklearn
import tensorflow as tf
import tensorflow_decision_forests as tfdf

drive.mount("/content/drive")

print("=" * 72)
print("CELL 01 — ENVIRONMENT AND REPRODUCIBILITY VERIFICATION")
print("=" * 72)

PROJECT_ROOT = Path("/content/drive/MyDrive/MSc_Project")

NSL10_DATA_DIR = PROJECT_ROOT / "data/NSL_KDD/processed"
NSL10_PREPROCESSING_DIR = (
    PROJECT_ROOT / "results/NSL_KDD_PREPROCESSING"
)
NSL10_RESULTS_DIR = (
    PROJECT_ROOT / "results/NSL_KDD_RF_REPLICATION"
)

NSL10_EXPECTED_FINGERPRINT = (
    "f321a798f07f753bf0de9455e423175590fd1b000259d3d698a83132aa4f5364"
)

NSL10_MANIFEST_PATH = (
    NSL10_PREPROCESSING_DIR / "NSL09_manifest.json"
)

NSL10_COMPLETION_PATH = (
    NSL10_PREPROCESSING_DIR / "NSL09_NOTEBOOK_COMPLETE.json"
)

print("\nENVIRONMENT")
print("Python:", sys.version.split()[0])
print("TensorFlow:", tf.__version__)
print("TFDF:", tfdf.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("PyArrow:", pyarrow.__version__)

if not hasattr(tfdf.keras, "RandomForestModel"):
    raise RuntimeError(
        "TFDF RandomForestModel is unavailable."
    )

if not NSL10_MANIFEST_PATH.is_file():
    raise FileNotFoundError(NSL10_MANIFEST_PATH)

if not NSL10_COMPLETION_PATH.is_file():
    raise FileNotFoundError(NSL10_COMPLETION_PATH)

NSL10_MANIFEST = json.loads(
    NSL10_MANIFEST_PATH.read_text(encoding="utf-8")
)

NSL10_PREPROCESSING_COMPLETION = json.loads(
    NSL10_COMPLETION_PATH.read_text(encoding="utf-8")
)

if NSL10_PREPROCESSING_COMPLETION["status"] != "COMPLETE":
    raise RuntimeError(
        "Notebook 09 completion record is invalid."
    )

if not NSL10_PREPROCESSING_COMPLETION["audit_checks_passed"]:
    raise RuntimeError(
        "Notebook 09 final audit was not verified."
    )

if (
    NSL10_MANIFEST["combined_artefact_fingerprint"]
    != NSL10_EXPECTED_FINGERPRINT
):
    raise RuntimeError(
        "NSL-KDD preprocessing fingerprint mismatch."
    )

def ns10_sha256(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(8 * 1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

# Verify original source files

for name, record in NSL10_MANIFEST["source_files"].items():

    path = Path(record["path"])

    if not path.is_file():
        raise FileNotFoundError(path)

    if ns10_sha256(path) != record["sha256"]:
        raise RuntimeError(
            f"Original source fingerprint mismatch: {name}"
        )

# Verify saved preprocessing artefacts

saved_artefacts = NSL10_MANIFEST["processed_artefacts"]

if len(saved_artefacts) != 10:
    raise RuntimeError(
        "Expected ten preprocessing artefacts."
    )

for relative_path, record in saved_artefacts.items():

    path = PROJECT_ROOT / relative_path

    if not path.is_file():
        raise FileNotFoundError(path)

    if ns10_sha256(path) != record["sha256"]:
        raise RuntimeError(
            f"Processed artefact fingerprint mismatch: "
            f"{relative_path}"
        )

# Verify the combined fingerprint

fingerprint_content = json.dumps(
    {
        name: record["sha256"]
        for name, record in saved_artefacts.items()
    },
    sort_keys=True,
    separators=(",", ":")
).encode("utf-8")

computed_fingerprint = hashlib.sha256(
    fingerprint_content
).hexdigest()

if computed_fingerprint != NSL10_EXPECTED_FINGERPRINT:
    raise RuntimeError(
        "Combined fingerprint verification failed."
    )

NSL10_INPUTS_READY = True

print("\nSOURCE VERIFICATION")
print(
    "Original source files:",
    len(NSL10_MANIFEST["source_files"])
)
print("Processed artefacts:", len(saved_artefacts))
print("Combined fingerprint:", computed_fingerprint)
print("Notebook 09 status: COMPLETE")
print("Source integrity: PASS")
print("Artefact integrity: PASS")

print("\nCELL 01: PASSED")
print("Model training performed: NO")
print("=" * 72)


Mounted at /content/drive
CELL 01 — ENVIRONMENT AND REPRODUCIBILITY VERIFICATION

ENVIRONMENT
Python: 3.12.13
TensorFlow: 2.19.0
TFDF: 1.12.0
NumPy: 2.0.2
Pandas: 2.2.2
Scikit-learn: 1.6.1
PyArrow: 18.1.0

SOURCE VERIFICATION
Original source files: 2
Processed artefacts: 10
Combined fingerprint: f321a798f07f753bf0de9455e423175590fd1b000259d3d698a83132aa4f5364
Notebook 09 status: COMPLETE
Source integrity: PASS
Artefact integrity: PASS

CELL 01: PASSED
Model training performed: NO


In [ ]:

# ============================================================
# NOTEBOOK 10 — NSL-KDD RANDOM FOREST REPLICATION
# CELL 02 — PROCESSED DATASET LOADING AND VALIDATION
# ============================================================

import json
import numpy as np
import pandas as pd

print("=" * 72)
print("CELL 02 — PROCESSED DATASET LOADING")
print("=" * 72)

if not globals().get("NSL10_INPUTS_READY", False):
    raise RuntimeError("Cell 01 must pass first.")

NSL10_SCHEMA = json.loads(
    (
        NSL10_DATA_DIR / "nslkdd_feature_schema.json"
    ).read_text(encoding="utf-8")
)

NSL10_CLASS_MAPPING = json.loads(
    (
        NSL10_DATA_DIR / "nslkdd_class_mapping.json"
    ).read_text(encoding="utf-8")
)

NSL10_FEATURE_NAMES = NSL10_SCHEMA["feature_names"]
NSL10_CLASS_NAMES = NSL10_CLASS_MAPPING["class_order"]

if len(NSL10_FEATURE_NAMES) != 122:
    raise RuntimeError("Expected 122 encoded predictors.")

if len(set(NSL10_FEATURE_NAMES)) != 122:
    raise RuntimeError("Duplicate predictor names.")

if NSL10_CLASS_NAMES != [
    "Normal", "DoS", "Probe", "R2L", "U2R"
]:
    raise RuntimeError("Unexpected target class order.")

expected_class_mapping = {
    name: index
    for index, name in enumerate(NSL10_CLASS_NAMES)
}

if (
    NSL10_CLASS_MAPPING["class_to_index"]
    != expected_class_mapping
):
    raise RuntimeError("Class index mapping mismatch.")

partition_files = {
    "train": "nslkdd_train.parquet",
    "validation": "nslkdd_validation.parquet",
    "test": "nslkdd_test.parquet"
}

expected_rows = {
    "train": 107077,
    "validation": 18896,
    "test": 22544
}

expected_class_counts = {
    "train": [57241, 39038, 9908, 846, 44],
    "validation": [10102, 6889, 1748, 149, 8],
    "test": [9711, 7460, 2421, 2885, 67]
}

NSL10_PARTITIONS = {}
NSL10_CLASS_COUNTS = {}

for name, filename in partition_files.items():

    path = NSL10_DATA_DIR / filename

    if not path.is_file():
        raise FileNotFoundError(path)

    frame = pd.read_parquet(path)

    expected_columns = NSL10_FEATURE_NAMES + ["Label"]

    if list(frame.columns) != expected_columns:
        raise RuntimeError(
            f"{name}: saved feature order mismatch."
        )

    if frame.shape != (expected_rows[name], 123):
        raise RuntimeError(
            f"{name}: unexpected dataset shape."
        )

    feature_frame = frame.loc[:, NSL10_FEATURE_NAMES]

    if not all(
        dtype == np.dtype("float32")
        for dtype in feature_frame.dtypes
    ):
        raise RuntimeError(
            f"{name}: expected float32 predictors."
        )

    if frame["Label"].dtype != np.dtype("int8"):
        raise RuntimeError(
            f"{name}: expected int8 target labels."
        )

    if frame.isna().any().any():
        raise RuntimeError(
            f"{name}: missing values detected."
        )

    X = feature_frame.to_numpy(
        dtype=np.float32,
        copy=False
    )

    y = frame["Label"].to_numpy(
        dtype=np.int32,
        copy=True
    )

    if not np.isfinite(X).all():
        raise RuntimeError(
            f"{name}: non-finite predictor values."
        )

    if np.any(y < 0) or np.any(y > 4):
        raise RuntimeError(
            f"{name}: invalid encoded class labels."
        )

    counts = np.bincount(y, minlength=5)

    if not np.array_equal(
        counts,
        expected_class_counts[name]
    ):
        raise RuntimeError(
            f"{name}: target class distribution mismatch."
        )

    NSL10_PARTITIONS[name] = frame
    NSL10_CLASS_COUNTS[name] = counts

print("\nPROCESSED DATASET SUMMARY")

for name, frame in NSL10_PARTITIONS.items():
    print(
        f"{name:<12} "
        f"Records: {len(frame):<8} "
        f"Predictors: {len(NSL10_FEATURE_NAMES):<4} "
        f"Classes: {len(np.unique(frame['Label']))}"
    )

NSL10_DISTRIBUTION = pd.DataFrame(
    NSL10_CLASS_COUNTS,
    index=NSL10_CLASS_NAMES
)

NSL10_DISTRIBUTION.index.name = "Class"

print("\nCLASS DISTRIBUTION")
print(NSL10_DISTRIBUTION.to_string())

print("\nSCHEMA VERIFICATION")
print("Encoded predictors:", len(NSL10_FEATURE_NAMES))
print("Target column:", NSL10_SCHEMA["label_column"])
print("Class order:", NSL10_CLASS_NAMES)
print("Feature dtype:", "float32")
print("Training and validation sources:", "KDDTrain+")
print("Official test source:", "KDDTest+")

NSL10_DATA_READY = True

print("\nCELL 02: PASSED")
print("Dataset loading: PASS")
print("Feature schema: PASS")
print("Class distributions: PASS")
print("Model training performed: NO")
print("=" * 72)


CELL 02 — PROCESSED DATASET LOADING

PROCESSED DATASET SUMMARY
train        Records: 107077   Predictors: 122  Classes: 5
validation   Records: 18896    Predictors: 122  Classes: 5
test         Records: 22544    Predictors: 122  Classes: 5

CLASS DISTRIBUTION
        train  validation  test
Class                          
Normal  57241       10102  9711
DoS     39038        6889  7460
Probe    9908        1748  2421
R2L       846         149  2885
U2R        44           8    67

SCHEMA VERIFICATION
Encoded predictors: 122
Target column: Label
Class order: ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']
Feature dtype: float32
Training and validation sources: KDDTrain+
Official test source: KDDTest+

CELL 02: PASSED
Dataset loading: PASS
Feature schema: PASS
Class distributions: PASS
Model training performed: NO


In [ ]:

# ============================================================
# NOTEBOOK 10 — NSL-KDD RANDOM FOREST REPLICATION
# CELL 03 — LOCKED RANDOM FOREST CONFIGURATION
# ============================================================

from pathlib import Path
import inspect
import json
import tensorflow_decision_forests as tfdf

print("=" * 72)
print("CELL 03 — LOCKED RANDOM FOREST CONFIGURATION")
print("=" * 72)

if not globals().get("NSL10_DATA_READY", False):
    raise RuntimeError("Cell 02 must pass first.")

NSL10_RANDOM_SEED = 42

NSL10_MODEL_CONFIGS = {
    "RF-B-NSL": {
        "num_trees": 100,
        "max_depth": -1,
        "min_examples": 1,
        "random_seed": NSL10_RANDOM_SEED
    },
    "RF-H2-NSL": {
        "num_trees": 100,
        "max_depth": 30,
        "min_examples": 1,
        "random_seed": NSL10_RANDOM_SEED
    }
}

NSL10_MODEL_PATHS = {
    "RF-B-NSL": (
        PROJECT_ROOT / "models/RF_B_NSL/saved_model"
    ),
    "RF-H2-NSL": (
        PROJECT_ROOT / "models/RF_H2_NSL/saved_model"
    )
}

NSL10_MODEL_RESULT_DIRS = {
    "RF-B-NSL": (
        NSL10_RESULTS_DIR / "RF_B_NSL"
    ),
    "RF-H2-NSL": (
        NSL10_RESULTS_DIR / "RF_H2_NSL"
    )
}

signature = inspect.signature(
    tfdf.keras.RandomForestModel
)

required_parameters = [
    "num_trees",
    "max_depth",
    "min_examples",
    "random_seed"
]

missing_parameters = [
    parameter
    for parameter in required_parameters
    if parameter not in signature.parameters
]

if missing_parameters:
    raise RuntimeError(
        "Required TFDF parameters unavailable: "
        + ", ".join(missing_parameters)
    )

baseline = NSL10_MODEL_CONFIGS["RF-B-NSL"]
tuned = NSL10_MODEL_CONFIGS["RF-H2-NSL"]

if baseline != {
    "num_trees": 100,
    "max_depth": -1,
    "min_examples": 1,
    "random_seed": 42
}:
    raise RuntimeError(
        "Baseline configuration mismatch."
    )

if tuned != {
    "num_trees": 100,
    "max_depth": 30,
    "min_examples": 1,
    "random_seed": 42
}:
    raise RuntimeError(
        "Frozen tuned configuration mismatch."
    )

if set(baseline) != set(tuned):
    raise RuntimeError(
        "Model hyperparameter sets differ."
    )

differing_parameters = [
    key for key in baseline
    if baseline[key] != tuned[key]
]

if differing_parameters != ["max_depth"]:
    raise RuntimeError(
        "Unexpected difference between model configurations."
    )

print("\nMODEL CONFIGURATIONS")

for model_name, config in NSL10_MODEL_CONFIGS.items():

    print("\n" + "-" * 72)
    print(model_name)
    print("-" * 72)

    print("Model type: RANDOM_FOREST")
    print("Trees:", config["num_trees"])
    print(
        "Maximum depth:",
        "Unrestricted"
        if config["max_depth"] == -1
        else config["max_depth"]
    )
    print("Minimum examples:", config["min_examples"])
    print("Random seed:", config["random_seed"])
    print("Input features:", len(NSL10_FEATURE_NAMES))
    print("Target classes:", len(NSL10_CLASS_NAMES))
    print("Training partition:", "Development Train")
    print("Validation partition:", "Development Validation")
    print("Test partition:", "Official NSL-KDD Test")
    print("Model destination:", NSL10_MODEL_PATHS[model_name])

print("\nCONFIGURATION CONSISTENCY")
print("Parameter differences:", differing_parameters)
print("Hyperparameter tuning:", "Disabled")
print("Feature selection:", "Disabled")
print("Synthetic resampling:", "Disabled")
print("CIC model transfer:", "Disabled")
print("Validation-derived configuration changes:", "Disabled")

# Prevent accidental modification of existing model artefacts.

existing_model_paths = [
    str(path)
    for path in NSL10_MODEL_PATHS.values()
    if path.exists()
]

if existing_model_paths:
    print("\nMODEL DESTINATION REVIEW")

    for path in existing_model_paths:
        print("Existing destination:", path)

    raise RuntimeError(
        "One or more model destinations already exist. "
        "Verify existing artefacts before training."
    )

NSL10_CONFIGURATION_LOCKED = True

print("\nCELL 03: PASSED")
print("Dataset configuration: VERIFIED")
print("Baseline configuration: LOCKED")
print("Tuned configuration: LOCKED")
print("Existing model destinations: CLEAR")
print("Model training performed: NO")
print("=" * 72)


CELL 03 — LOCKED RANDOM FOREST CONFIGURATION

MODEL CONFIGURATIONS

------------------------------------------------------------------------
RF-B-NSL
------------------------------------------------------------------------
Model type: RANDOM_FOREST
Trees: 100
Maximum depth: Unrestricted
Minimum examples: 1
Random seed: 42
Input features: 122
Target classes: 5
Training partition: Development Train
Validation partition: Development Validation
Test partition: Official NSL-KDD Test
Model destination: /content/drive/MyDrive/MSc_Project/models/RF_B_NSL/saved_model

------------------------------------------------------------------------
RF-H2-NSL
------------------------------------------------------------------------
Model type: RANDOM_FOREST
Trees: 100
Maximum depth: 30
Minimum examples: 1
Random seed: 42
Input features: 122
Target classes: 5
Training partition: Development Train
Validation partition: Development Validation
Test partition: Official NSL-KDD Test
Model destination: /content/

In [ ]:

# ============================================================
# NOTEBOOK 10 — NSL-KDD RANDOM FOREST REPLICATION
# CELL 04 — TFDF DATASET PREPARATION
# ============================================================

import re
import numpy as np
import tensorflow as tf
import tensorflow_decision_forests as tfdf

print("=" * 72)
print("CELL 04 — TFDF DATASET PREPARATION")
print("=" * 72)

if not globals().get("NSL10_CONFIGURATION_LOCKED", False):
    raise RuntimeError("Cell 03 must pass first.")

NSL10_TF_DATA_READY = False

if len(NSL10_FEATURE_NAMES) != 122:
    raise RuntimeError("Expected 122 predictors.")

if NSL10_CLASS_NAMES != [
    "Normal", "DoS", "Probe", "R2L", "U2R"
]:
    raise RuntimeError("Class mapping mismatch.")

# Verify that encoded names can be used unchanged.

invalid_feature_names = [
    name for name in NSL10_FEATURE_NAMES
    if re.fullmatch(r"[A-Za-z0-9_]+", name) is None
]

if invalid_feature_names:
    raise RuntimeError(
        "Unsupported TFDF feature names: "
        + str(invalid_feature_names[:10])
    )

NSL10_TF_BATCH_SIZE = 1024

NSL10_TF_DATASETS = {}

for partition_name in ["train", "validation"]:

    frame = NSL10_PARTITIONS[partition_name]

    if list(frame.columns) != (
        NSL10_FEATURE_NAMES + ["Label"]
    ):
        raise RuntimeError(
            f"{partition_name}: unexpected feature schema."
        )

    if frame["Label"].isna().any():
        raise RuntimeError(
            f"{partition_name}: missing target values."
        )

    dataset = tfdf.keras.pd_dataframe_to_tf_dataset(
        frame,
        label="Label",
        task=tfdf.keras.Task.CLASSIFICATION,
        batch_size=NSL10_TF_BATCH_SIZE,
        fix_feature_names=False,
        in_place=False
    )

    NSL10_TF_DATASETS[partition_name] = dataset

    # Verify the first batch against the original partition.

    features, labels = next(iter(dataset))

    if len(features) != 122:
        raise RuntimeError(
            f"{partition_name}: unexpected feature count."
        )

    if set(features) != set(NSL10_FEATURE_NAMES):
        raise RuntimeError(
            f"{partition_name}: TFDF feature names differ."
        )

    if not labels.dtype.is_integer:
        raise RuntimeError(
            f"{partition_name}: expected integer class labels."
        )

    batch_length = int(labels.shape[0])

    expected_labels = frame["Label"].iloc[
        :batch_length
    ].to_numpy(dtype=np.int32)

    if not np.array_equal(
        labels.numpy().reshape(-1),
        expected_labels
    ):
        raise RuntimeError(
            f"{partition_name}: batch label alignment failed."
        )

    for feature_name in NSL10_FEATURE_NAMES:

        tensor = features[feature_name]

        if tensor.dtype != tf.float32:
            raise RuntimeError(
                f"{partition_name}: invalid feature dtype "
                f"for {feature_name}."
            )

        actual_values = tensor.numpy().reshape(-1)

        expected_values = frame[
            feature_name
        ].iloc[:batch_length].to_numpy(dtype=np.float32)

        if not np.array_equal(
            actual_values,
            expected_values
        ):
            raise RuntimeError(
                f"{partition_name}: feature alignment failed "
                f"for {feature_name}."
            )

    print("\n" + "-" * 72)
    print(partition_name.upper())
    print("-" * 72)

    print("Records:", len(frame))
    print("Input features:", len(features))
    print("Batch size:", NSL10_TF_BATCH_SIZE)
    print("First batch records:", batch_length)
    print("Feature dtype:", "float32")
    print("Label dtype:", labels.dtype.name)
    print("Feature and label alignment: PASS")

# Verify class representation.

for name in ["train", "validation"]:

    target = NSL10_PARTITIONS[name][
        "Label"
    ].to_numpy(dtype=np.int32)

    if not np.array_equal(
        np.unique(target),
        np.arange(5)
    ):
        raise RuntimeError(
            f"{name}: incomplete class representation."
        )

NSL10_TF_DATA_READY = True

print("\nDATASET VERIFICATION")
print("Training dataset: READY")
print("Validation dataset: READY")
print("Training records: 107077")
print("Validation records: 18896")
print("Encoded features: 122")
print("Target classes: 5")
print("Original class-index mapping: PRESERVED")
print("Official test accessed: NO")

print("\nCELL 04: PASSED")
print("Model training performed: NO")
print("=" * 72)


CELL 04 — TFDF DATASET PREPARATION

------------------------------------------------------------------------
TRAIN
------------------------------------------------------------------------
Records: 107077
Input features: 122
Batch size: 1024
First batch records: 1024
Feature dtype: float32
Label dtype: int8
Feature and label alignment: PASS

------------------------------------------------------------------------
VALIDATION
------------------------------------------------------------------------
Records: 18896
Input features: 122
Batch size: 1024
First batch records: 1024
Feature dtype: float32
Label dtype: int8
Feature and label alignment: PASS

DATASET VERIFICATION
Training dataset: READY
Validation dataset: READY
Training records: 107077
Validation records: 18896
Encoded features: 122
Target classes: 5
Original class-index mapping: PRESERVED
Official test accessed: NO

CELL 04: PASSED
Model training performed: NO


In [ ]:

# ============================================================
# NOTEBOOK 10 — NSL-KDD RANDOM FOREST REPLICATION
# CELL 05 — BASELINE RANDOM FOREST TRAINING
# ============================================================

from pathlib import Path
import tempfile
import time
import numpy as np
import tensorflow as tf
import tensorflow_decision_forests as tfdf

print("=" * 72)
print("CELL 05 — BASELINE RANDOM FOREST TRAINING")
print("=" * 72)

if not globals().get("NSL10_TF_DATA_READY", False):
    raise RuntimeError("Cell 04 must pass first.")

NSL10_BASELINE_TRAINED = False

MODEL_NAME = "RF-B-NSL"

MODEL_CONFIG = NSL10_MODEL_CONFIGS[MODEL_NAME].copy()
MODEL_PATH = NSL10_MODEL_PATHS[MODEL_NAME]

if MODEL_CONFIG != {
    "num_trees": 100,
    "max_depth": -1,
    "min_examples": 1,
    "random_seed": 42
}:
    raise RuntimeError(
        "Baseline hyperparameter configuration mismatch."
    )

if MODEL_PATH.exists():
    raise RuntimeError(
        f"Model destination already exists: {MODEL_PATH}"
    )

MODEL_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

staging_path = (
    MODEL_PATH.parent / "saved_model_staging"
)

if staging_path.exists():
    raise RuntimeError(
        f"Existing staging directory: {staging_path}"
    )

training_temporary_directory = tempfile.mkdtemp(
    prefix="nsl10_rf_b_",
    dir="/content"
)

print("\nTRAINING CONFIGURATION")
print("Model:", MODEL_NAME)
print("Algorithm: RANDOM_FOREST")
print("Trees:", MODEL_CONFIG["num_trees"])
print("Maximum depth: Unrestricted")
print("Minimum examples:", MODEL_CONFIG["min_examples"])
print("Random seed:", MODEL_CONFIG["random_seed"])
print("Training records:", len(NSL10_PARTITIONS["train"]))
print("Input features:", len(NSL10_FEATURE_NAMES))
print("Classes:", len(NSL10_CLASS_NAMES))
print("Validation used for fitting: NO")
print("Official test used for fitting: NO")

# Initialise the baseline model.

NSL10_BASELINE_MODEL = tfdf.keras.RandomForestModel(
    task=tfdf.keras.Task.CLASSIFICATION,
    num_trees=MODEL_CONFIG["num_trees"],
    max_depth=MODEL_CONFIG["max_depth"],
    min_examples=MODEL_CONFIG["min_examples"],
    random_seed=MODEL_CONFIG["random_seed"],
    try_resume_training=False,
    temp_directory=training_temporary_directory,
    verbose=0,
    name="RF_B_NSL"
)

# Fit exclusively on development training data.

print("\nTRAINING")

training_start = time.perf_counter()

NSL10_BASELINE_MODEL.fit(
    NSL10_TF_DATASETS["train"],
    verbose=0
)

NSL10_BASELINE_TRAIN_SECONDS = (
    time.perf_counter() - training_start
)

# Inspect trained forest.

NSL10_BASELINE_INSPECTOR = (
    NSL10_BASELINE_MODEL.make_inspector()
)

actual_model_type = (
    NSL10_BASELINE_INSPECTOR.model_type()
)

actual_tree_count = (
    NSL10_BASELINE_INSPECTOR.num_trees()
)

if actual_model_type != "RANDOM_FOREST":
    raise RuntimeError(
        f"Unexpected model type: {actual_model_type}"
    )

if actual_tree_count != 100:
    raise RuntimeError(
        f"Unexpected trained tree count: {actual_tree_count}"
    )

# Verify encoded target metadata.

header = NSL10_BASELINE_INSPECTOR.header
dataspec = NSL10_BASELINE_INSPECTOR.dataspec

if callable(header):
    header = header()

if callable(dataspec):
    dataspec = dataspec()

label_column = dataspec.columns[
    header.label_col_idx
]

if not label_column.categorical.is_already_integerized:
    raise RuntimeError(
        "Model target is not integer-encoded."
    )

if label_column.categorical.number_of_unique_values != 6:
    raise RuntimeError(
        "Unexpected number of encoded target values."
    )

# Save model to a staging directory.

print("\nMODEL EXPORT")

NSL10_BASELINE_MODEL.save(
    str(staging_path),
    save_format="tf"
)

if not (staging_path / "saved_model.pb").is_file():
    raise RuntimeError(
        "SavedModel export verification failed."
    )

if not (staging_path / "assets").is_dir():
    raise RuntimeError(
        "SavedModel assets directory unavailable."
    )

# Finalise export only after structural verification.

if MODEL_PATH.exists():
    raise RuntimeError(
        "Final model destination became occupied."
    )

staging_path.rename(MODEL_PATH)

if not (MODEL_PATH / "saved_model.pb").is_file():
    raise RuntimeError(
        "Final SavedModel file unavailable."
    )

NSL10_BASELINE_TRAINED = True

print("\nTRAINING SUMMARY")
print("Model:", MODEL_NAME)
print("Model type:", actual_model_type)
print("Trained trees:", actual_tree_count)
print("Encoded target classes:", 5)
print("Training records:", 107077)
print(
    "Training time (seconds):",
    round(NSL10_BASELINE_TRAIN_SECONDS, 3)
)
print("SavedModel:", MODEL_PATH)
print("SavedModel verification: PASS")

# Report out-of-bag evaluation when available.

oob_evaluation = NSL10_BASELINE_INSPECTOR.evaluation()

print("\nOUT-OF-BAG EVALUATION")

if oob_evaluation is None:
    print("OOB evaluation: Unavailable")
else:
    print(
        "OOB accuracy:",
        getattr(oob_evaluation, "accuracy", None)
    )
    print(
        "OOB loss:",
        getattr(oob_evaluation, "loss", None)
    )

print("\nCELL 05: PASSED")
print("Baseline Random Forest trained: YES")
print("Baseline SavedModel exported: YES")
print("Validation evaluation: PENDING")
print("Official test evaluation: NOT PERFORMED")
print("=" * 72)


CELL 05 — BASELINE RANDOM FOREST TRAINING

TRAINING CONFIGURATION
Model: RF-B-NSL
Algorithm: RANDOM_FOREST
Trees: 100
Maximum depth: Unrestricted
Minimum examples: 1
Random seed: 42
Training records: 107077
Input features: 122
Classes: 5
Validation used for fitting: NO
Official test used for fitting: NO

TRAINING

MODEL EXPORT

TRAINING SUMMARY
Model: RF-B-NSL
Model type: RANDOM_FOREST
Trained trees: 100
Encoded target classes: 5
Training records: 107077
Training time (seconds): 24.534
SavedModel: /content/drive/MyDrive/MSc_Project/models/RF_B_NSL/saved_model
SavedModel verification: PASS

OUT-OF-BAG EVALUATION
OOB accuracy: 0.9989820409611775
OOB loss: 0.005541853605396244

CELL 05: PASSED
Baseline Random Forest trained: YES
Baseline SavedModel exported: YES
Validation evaluation: PENDING
Official test evaluation: NOT PERFORMED


In [ ]:

# ============================================================
# NOTEBOOK 10 — NSL-KDD RANDOM FOREST REPLICATION
# CELL 06 — BASELINE VALIDATION EVALUATION
# ============================================================

from pathlib import Path
import io
import json
import time
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    classification_report,
    confusion_matrix
)

print("=" * 72)
print("CELL 06 — BASELINE VALIDATION EVALUATION")
print("=" * 72)

if not globals().get("NSL10_BASELINE_TRAINED", False):
    raise RuntimeError("Cell 05 must pass first.")

MODEL_NAME = "RF-B-NSL"

MODEL_PATH = NSL10_MODEL_PATHS[MODEL_NAME]
RESULT_DIR = NSL10_MODEL_RESULT_DIRS[MODEL_NAME]

NSL10_BASELINE_VALIDATED = False

if not (MODEL_PATH / "saved_model.pb").is_file():
    raise FileNotFoundError(
        MODEL_PATH / "saved_model.pb"
    )

RESULT_DIR.mkdir(parents=True, exist_ok=True)

# Reload the saved model independently.

saved_model = tf.saved_model.load(
    str(MODEL_PATH)
)

if "serving_default" not in saved_model.signatures:
    raise RuntimeError(
        "SavedModel serving signature unavailable."
    )

signature = saved_model.signatures["serving_default"]

_, input_signature = (
    signature.structured_input_signature
)

if not input_signature:
    raise RuntimeError(
        "SavedModel has no named input features."
    )

if not set(input_signature).issubset(
    set(NSL10_FEATURE_NAMES)
):
    missing = sorted(
        set(input_signature)
        - set(NSL10_FEATURE_NAMES)
    )

    raise RuntimeError(
        f"SavedModel inputs not found in schema: {missing}"
    )

validation_frame = NSL10_PARTITIONS["validation"]

y_true = validation_frame[
    "Label"
].to_numpy(dtype=np.int32)

N = len(y_true)

if N != 18896:
    raise RuntimeError(
        "Unexpected validation observation count."
    )

# Verify output-class interpretation.

inspector = NSL10_BASELINE_INSPECTOR

header = inspector.header
dataspec = inspector.dataspec

if callable(header):
    header = header()

if callable(dataspec):
    dataspec = dataspec()

label_column = dataspec.columns[
    header.label_col_idx
]

if not label_column.categorical.is_already_integerized:
    raise RuntimeError(
        "Integer target encoding verification failed."
    )

if label_column.categorical.number_of_unique_values != 6:
    raise RuntimeError(
        "Unexpected target class cardinality."
    )

# Generate validation predictions using the saved signature.

batch_size = 4096
prediction_indices = np.empty(
    N,
    dtype=np.int32
)

prediction_start = time.perf_counter()

for offset in range(0, N, batch_size):

    end = min(offset + batch_size, N)

    batch_frame = validation_frame.iloc[
        offset:end
    ]

    input_tensors = {}

    for feature_name, specification in input_signature.items():

        if specification.dtype != tf.float32:
            raise RuntimeError(
                f"Unexpected input dtype: {feature_name}"
            )

        values = batch_frame[
            feature_name
        ].to_numpy(dtype=np.float32)

        input_tensors[feature_name] = (
            tf.convert_to_tensor(
                values,
                dtype=tf.float32
            )
        )

    outputs = signature(**input_tensors)

    if len(outputs) != 1:
        raise RuntimeError(
            "Unexpected SavedModel output structure."
        )

    probabilities = next(
        iter(outputs.values())
    ).numpy()

    if probabilities.shape != (end - offset, 5):
        raise RuntimeError(
            f"Unexpected prediction shape: "
            f"{probabilities.shape}"
        )

    if not np.isfinite(probabilities).all():
        raise RuntimeError(
            "Non-finite prediction probabilities."
        )

    if np.any(probabilities < -1e-6):
        raise RuntimeError(
            "Invalid negative class probabilities."
        )

    if not np.allclose(
        probabilities.sum(axis=1),
        1.0,
        atol=1e-4,
        rtol=0
    ):
        raise RuntimeError(
            "Prediction probabilities do not sum to one."
        )

    prediction_indices[offset:end] = np.argmax(
        probabilities,
        axis=1
    )

inference_seconds = (
    time.perf_counter() - prediction_start
)

if not np.array_equal(
    np.unique(y_true),
    np.arange(5)
):
    raise RuntimeError(
        "Unexpected validation target classes."
    )

if np.any(prediction_indices < 0) or np.any(
    prediction_indices >= 5
):
    raise RuntimeError(
        "Predicted class indices are invalid."
    )

NSL10_BASELINE_VAL_TRUE = y_true
NSL10_BASELINE_VAL_PRED = prediction_indices

labels = list(range(5))

metrics = {
    "Accuracy": accuracy_score(
        y_true, prediction_indices
    ),
    "Macro Precision": precision_score(
        y_true, prediction_indices,
        labels=labels,
        average="macro",
        zero_division=0
    ),
    "Macro Recall": recall_score(
        y_true, prediction_indices,
        labels=labels,
        average="macro",
        zero_division=0
    ),
    "Macro F1": f1_score(
        y_true, prediction_indices,
        labels=labels,
        average="macro",
        zero_division=0
    ),
    "Weighted F1": f1_score(
        y_true, prediction_indices,
        labels=labels,
        average="weighted",
        zero_division=0
    ),
    "Balanced Accuracy": balanced_accuracy_score(
        y_true, prediction_indices
    ),
    "MCC": matthews_corrcoef(
        y_true, prediction_indices
    )
}

report_text = classification_report(
    y_true,
    prediction_indices,
    labels=labels,
    target_names=NSL10_CLASS_NAMES,
    digits=6,
    zero_division=0
)

report_dict = classification_report(
    y_true,
    prediction_indices,
    labels=labels,
    target_names=NSL10_CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(
    report_dict
).transpose()

cm = confusion_matrix(
    y_true,
    prediction_indices,
    labels=labels
)

cm_df = pd.DataFrame(
    cm,
    index=NSL10_CLASS_NAMES,
    columns=NSL10_CLASS_NAMES
)

if cm.shape != (5, 5):
    raise RuntimeError(
        "Unexpected confusion matrix shape."
    )

if cm.sum() != N:
    raise RuntimeError(
        "Confusion matrix total mismatch."
    )

if int(np.trace(cm)) != int(
    np.sum(prediction_indices == y_true)
):
    raise RuntimeError(
        "Confusion matrix correct-count mismatch."
    )

if not np.isclose(
    metrics["Macro F1"],
    report_df.loc["macro avg", "f1-score"],
    atol=1e-12,
    rtol=0
):
    raise RuntimeError(
        "Macro F1 consistency check failed."
    )

# Preserve validation outputs without overwriting prior files.

def ns10_write_once(path, content):

    path = Path(path)

    try:
        with path.open("xb") as handle:
            handle.write(content)
        return "CREATED"

    except FileExistsError:

        if path.read_bytes() == content:
            return "ALREADY VERIFIED"

        raise RuntimeError(
            f"Existing result differs: {path}"
        )

metrics_record = {
    "Model": MODEL_NAME,
    "Partition": "validation",
    "Records": N,
    **metrics
}

metrics_bytes = pd.DataFrame(
    [metrics_record]
).to_csv(
    index=False,
    float_format="%.9f"
).encode("utf-8")

report_bytes = report_df.to_csv(
    index_label="Class",
    float_format="%.9f"
).encode("utf-8")

cm_bytes = cm_df.to_csv(
    index_label="Actual Class"
).encode("utf-8")

prediction_buffer = io.BytesIO()

np.save(
    prediction_buffer,
    prediction_indices.astype(np.int8),
    allow_pickle=False
)

print("\nRESULT EXPORTS")

exports = {
    "RF_B_NSL_validation_metrics.csv": metrics_bytes,
    "RF_B_NSL_validation_classification_report.csv": report_bytes,
    "RF_B_NSL_validation_confusion_matrix.csv": cm_bytes,
    "RF_B_NSL_validation_predictions.npy": (
        prediction_buffer.getvalue()
    )
}

for filename, content in exports.items():
    status = ns10_write_once(
        RESULT_DIR / filename,
        content
    )

    print(f"{filename}: {status}")

NSL10_BASELINE_VALIDATION_METRICS = metrics.copy()
NSL10_BASELINE_VALIDATED = True

print("\nVALIDATION METRICS")

for name, value in metrics.items():
    print(f"{name:<22}: {value:.6f}")

print("\nCLASSIFICATION REPORT")
print(report_text)

print("CONFUSION MATRIX")
print(cm_df.to_string())

print("\nEVALUATION SUMMARY")
print("Model:", MODEL_NAME)
print("SavedModel reloaded: PASS")
print("Serving input features:", len(input_signature))
print("Prediction observations:", N)
print("Class-index range:", "0–4")
print("Classification report: PASS")
print("Confusion matrix: PASS")
print("Validation result exports:", len(exports))
print(
    "Inference time (seconds):",
    round(inference_seconds, 3)
)

print("\nCELL 06: PASSED")
print("Baseline validation evaluation: COMPLETE")
print("Official test evaluation: NOT PERFORMED")
print("=" * 72)


CELL 06 — BASELINE VALIDATION EVALUATION

RESULT EXPORTS
RF_B_NSL_validation_metrics.csv: CREATED
RF_B_NSL_validation_classification_report.csv: CREATED
RF_B_NSL_validation_confusion_matrix.csv: CREATED
RF_B_NSL_validation_predictions.npy: CREATED

VALIDATION METRICS
Accuracy              : 0.999312
Macro Precision       : 0.971182
Macro Recall          : 0.946560
Macro F1              : 0.958147
Weighted F1           : 0.999306
Balanced Accuracy     : 0.946560
MCC                   : 0.998799

CLASSIFICATION REPORT
              precision    recall  f1-score   support

      Normal   0.998912  0.999802  0.999357     10102
         DoS   0.999855  0.999855  0.999855      6889
       Probe   1.000000  0.996568  0.998281      1748
         R2L   1.000000  0.986577  0.993243       149
         U2R   0.857143  0.750000  0.800000         8

    accuracy                       0.999312     18896
   macro avg   0.971182  0.946560  0.958147     18896
weighted avg   0.999305  0.999312  0.999306 

In [ ]:

# ============================================================
# NOTEBOOK 10 — NSL-KDD RANDOM FOREST REPLICATION
# CELL 07 — FROZEN-TUNING RANDOM FOREST TRAINING
# ============================================================

from pathlib import Path
import tempfile
import time

import tensorflow as tf
import tensorflow_decision_forests as tfdf

print("=" * 72)
print("CELL 07 — FROZEN-TUNING RANDOM FOREST TRAINING")
print("=" * 72)

if not globals().get("NSL10_CONFIGURATION_LOCKED", False):
    raise RuntimeError("Cell 03 must pass first.")

if not globals().get("NSL10_TF_DATA_READY", False):
    raise RuntimeError("Cell 04 must pass first.")

if not globals().get("NSL10_BASELINE_VALIDATED", False):
    raise RuntimeError("Cell 06 must pass first.")

NSL10_TUNED_TRAINED = False

MODEL_NAME = "RF-H2-NSL"

MODEL_CONFIG = NSL10_MODEL_CONFIGS[MODEL_NAME].copy()
MODEL_PATH = NSL10_MODEL_PATHS[MODEL_NAME]

EXPECTED_CONFIG = {
    "num_trees": 100,
    "max_depth": 30,
    "min_examples": 1,
    "random_seed": 42
}

if MODEL_CONFIG != EXPECTED_CONFIG:
    raise RuntimeError(
        "Frozen tuning configuration mismatch."
    )

baseline_config = NSL10_MODEL_CONFIGS["RF-B-NSL"]

if any(
    MODEL_CONFIG[key] != baseline_config[key]
    for key in ["num_trees", "min_examples", "random_seed"]
):
    raise RuntimeError(
        "Unexpected configuration difference from baseline."
    )

if MODEL_PATH.exists():
    raise RuntimeError(
        f"Model destination already exists: {MODEL_PATH}"
    )

if len(NSL10_PARTITIONS["train"]) != 107077:
    raise RuntimeError(
        "Unexpected development training size."
    )

MODEL_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

staging_path = (
    MODEL_PATH.parent / "saved_model_staging"
)

if staging_path.exists():
    raise RuntimeError(
        f"Existing staging directory: {staging_path}"
    )

temporary_directory = tempfile.mkdtemp(
    prefix="nsl10_rf_h2_",
    dir="/content"
)

print("\nTRAINING CONFIGURATION")
print("Model:", MODEL_NAME)
print("Algorithm: RANDOM_FOREST")
print("Trees:", MODEL_CONFIG["num_trees"])
print("Maximum depth:", MODEL_CONFIG["max_depth"])
print("Minimum examples:", MODEL_CONFIG["min_examples"])
print("Random seed:", MODEL_CONFIG["random_seed"])
print("Training records:", len(NSL10_PARTITIONS["train"]))
print("Input features:", len(NSL10_FEATURE_NAMES))
print("Target classes:", len(NSL10_CLASS_NAMES))
print("Validation used for fitting: NO")
print("Official test used for fitting: NO")
print("Hyperparameter search: DISABLED")
print("Resampling: DISABLED")

# Initialise the frozen-configuration model.

NSL10_TUNED_MODEL = tfdf.keras.RandomForestModel(
    task=tfdf.keras.Task.CLASSIFICATION,
    num_trees=MODEL_CONFIG["num_trees"],
    max_depth=MODEL_CONFIG["max_depth"],
    min_examples=MODEL_CONFIG["min_examples"],
    random_seed=MODEL_CONFIG["random_seed"],
    try_resume_training=False,
    temp_directory=temporary_directory,
    verbose=0,
    name="RF_H2_NSL"
)

# Fit only on development training records.

print("\nTRAINING")

training_start = time.perf_counter()

NSL10_TUNED_MODEL.fit(
    NSL10_TF_DATASETS["train"],
    verbose=0
)

NSL10_TUNED_TRAIN_SECONDS = (
    time.perf_counter() - training_start
)

# Verify trained forest structure.

NSL10_TUNED_INSPECTOR = (
    NSL10_TUNED_MODEL.make_inspector()
)

actual_model_type = (
    NSL10_TUNED_INSPECTOR.model_type()
)

actual_tree_count = (
    NSL10_TUNED_INSPECTOR.num_trees()
)

if actual_model_type != "RANDOM_FOREST":
    raise RuntimeError(
        f"Unexpected model type: {actual_model_type}"
    )

if actual_tree_count != 100:
    raise RuntimeError(
        f"Unexpected tree count: {actual_tree_count}"
    )

header = NSL10_TUNED_INSPECTOR.header
dataspec = NSL10_TUNED_INSPECTOR.dataspec

if callable(header):
    header = header()

if callable(dataspec):
    dataspec = dataspec()

label_column = dataspec.columns[
    header.label_col_idx
]

if not label_column.categorical.is_already_integerized:
    raise RuntimeError(
        "Integer class encoding verification failed."
    )

if label_column.categorical.number_of_unique_values != 6:
    raise RuntimeError(
        "Unexpected encoded class count."
    )

# Export to an isolated staging directory.

print("\nMODEL EXPORT")

NSL10_TUNED_MODEL.save(
    str(staging_path),
    save_format="tf"
)

if not (staging_path / "saved_model.pb").is_file():
    raise RuntimeError(
        "SavedModel export file unavailable."
    )

if not (staging_path / "assets").is_dir():
    raise RuntimeError(
        "SavedModel assets directory unavailable."
    )

# Validate the SavedModel serving interface.

staged_model = tf.saved_model.load(
    str(staging_path)
)

if "serving_default" not in staged_model.signatures:
    raise RuntimeError(
        "SavedModel serving signature unavailable."
    )

staged_signature = (
    staged_model.signatures["serving_default"]
)

_, staged_inputs = (
    staged_signature.structured_input_signature
)

if set(staged_inputs) != set(NSL10_FEATURE_NAMES):
    raise RuntimeError(
        "SavedModel feature schema mismatch."
    )

if len(staged_signature.structured_outputs) != 1:
    raise RuntimeError(
        "Unexpected SavedModel output structure."
    )

output_specification = next(
    iter(staged_signature.structured_outputs.values())
)

if output_specification.shape[-1] != 5:
    raise RuntimeError(
        "SavedModel output class count mismatch."
    )

# Finalise the model directory.

if MODEL_PATH.exists():
    raise RuntimeError(
        "Final model destination became occupied."
    )

staging_path.rename(MODEL_PATH)

if not (MODEL_PATH / "saved_model.pb").is_file():
    raise RuntimeError(
        "Final SavedModel verification failed."
    )

NSL10_TUNED_TRAINED = True

print("\nTRAINING SUMMARY")
print("Model:", MODEL_NAME)
print("Model type:", actual_model_type)
print("Trained trees:", actual_tree_count)
print("Maximum depth configuration:", MODEL_CONFIG["max_depth"])
print("Encoded target classes:", 5)
print("Training records:", 107077)
print(
    "Training time (seconds):",
    round(NSL10_TUNED_TRAIN_SECONDS, 3)
)
print("SavedModel:", MODEL_PATH)
print("Serving features:", len(staged_inputs))
print("Serving output classes:", output_specification.shape[-1])
print("SavedModel verification: PASS")

# Report out-of-bag evaluation.

print("\nOUT-OF-BAG EVALUATION")

oob_evaluation = NSL10_TUNED_INSPECTOR.evaluation()

if oob_evaluation is None:
    print("OOB evaluation: Unavailable")
else:
    print(
        "OOB accuracy:",
        getattr(oob_evaluation, "accuracy", None)
    )
    print(
        "OOB loss:",
        getattr(oob_evaluation, "loss", None)
    )

print("\nCELL 07: PASSED")
print("Frozen-tuning Random Forest trained: YES")
print("SavedModel exported: YES")
print("Validation evaluation: PENDING")
print("Official test evaluation: NOT PERFORMED")
print("=" * 72)


CELL 07 — FROZEN-TUNING RANDOM FOREST TRAINING

TRAINING CONFIGURATION
Model: RF-H2-NSL
Algorithm: RANDOM_FOREST
Trees: 100
Maximum depth: 30
Minimum examples: 1
Random seed: 42
Training records: 107077
Input features: 122
Target classes: 5
Validation used for fitting: NO
Official test used for fitting: NO
Hyperparameter search: DISABLED
Resampling: DISABLED

TRAINING

MODEL EXPORT

TRAINING SUMMARY
Model: RF-H2-NSL
Model type: RANDOM_FOREST
Trained trees: 100
Maximum depth configuration: 30
Encoded target classes: 5
Training records: 107077
Training time (seconds): 19.834
SavedModel: /content/drive/MyDrive/MSc_Project/models/RF_H2_NSL/saved_model
Serving features: 122
Serving output classes: 5
SavedModel verification: PASS

OUT-OF-BAG EVALUATION
OOB accuracy: 0.9989820409611775
OOB loss: 0.005541853605396244

CELL 07: PASSED
Frozen-tuning Random Forest trained: YES
SavedModel exported: YES
Validation evaluation: PENDING
Official test evaluation: NOT PERFORMED


In [ ]:

# ============================================================
# NOTEBOOK 10 — NSL-KDD RANDOM FOREST REPLICATION
# CELL 08 — FROZEN-TUNING VALIDATION EVALUATION
# ============================================================

from pathlib import Path
import io
import time

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    classification_report,
    confusion_matrix
)

print("=" * 72)
print("CELL 08 — FROZEN-TUNING VALIDATION EVALUATION")
print("=" * 72)

if not globals().get("NSL10_TUNED_TRAINED", False):
    raise RuntimeError("Cell 07 must pass first.")

NSL10_TUNED_VALIDATED = False

MODEL_NAME = "RF-H2-NSL"

MODEL_PATH = NSL10_MODEL_PATHS[MODEL_NAME]
RESULT_DIR = NSL10_MODEL_RESULT_DIRS[MODEL_NAME]

if not (MODEL_PATH / "saved_model.pb").is_file():
    raise FileNotFoundError(
        MODEL_PATH / "saved_model.pb"
    )

RESULT_DIR.mkdir(parents=True, exist_ok=True)

# Independently load the saved model.

saved_model = tf.saved_model.load(
    str(MODEL_PATH)
)

if "serving_default" not in saved_model.signatures:
    raise RuntimeError(
        "SavedModel serving signature unavailable."
    )

signature = (
    saved_model.signatures["serving_default"]
)

_, input_signature = (
    signature.structured_input_signature
)

if set(input_signature) != set(NSL10_FEATURE_NAMES):
    raise RuntimeError(
        "SavedModel input feature schema mismatch."
    )

# Verify integer-encoded model targets.

inspector = NSL10_TUNED_INSPECTOR

header = inspector.header
dataspec = inspector.dataspec

if callable(header):
    header = header()

if callable(dataspec):
    dataspec = dataspec()

label_column = dataspec.columns[
    header.label_col_idx
]

if not label_column.categorical.is_already_integerized:
    raise RuntimeError(
        "Integer class encoding verification failed."
    )

if label_column.categorical.number_of_unique_values != 6:
    raise RuntimeError(
        "Unexpected model target cardinality."
    )

# Load the unchanged development validation partition.

validation_frame = NSL10_PARTITIONS["validation"]

y_true = validation_frame[
    "Label"
].to_numpy(dtype=np.int32)

N = len(y_true)

if N != 18896:
    raise RuntimeError(
        "Unexpected validation observation count."
    )

if not np.array_equal(
    y_true,
    NSL10_BASELINE_VAL_TRUE
):
    raise RuntimeError(
        "Baseline and tuned validation targets differ."
    )

# Generate predictions in fixed-size batches.

batch_size = 4096

prediction_indices = np.empty(
    N,
    dtype=np.int32
)

prediction_start = time.perf_counter()

for offset in range(0, N, batch_size):

    end = min(offset + batch_size, N)

    batch_frame = validation_frame.iloc[
        offset:end
    ]

    model_inputs = {}

    for feature_name, specification in input_signature.items():

        if specification.dtype != tf.float32:
            raise RuntimeError(
                f"Unexpected input dtype: {feature_name}"
            )

        values = batch_frame[
            feature_name
        ].to_numpy(dtype=np.float32)

        model_inputs[feature_name] = tf.convert_to_tensor(
            values,
            dtype=tf.float32
        )

    outputs = signature(**model_inputs)

    if len(outputs) != 1:
        raise RuntimeError(
            "Unexpected SavedModel output structure."
        )

    probabilities = next(
        iter(outputs.values())
    ).numpy()

    if probabilities.shape != (end - offset, 5):
        raise RuntimeError(
            f"Unexpected output shape: "
            f"{probabilities.shape}"
        )

    if not np.isfinite(probabilities).all():
        raise RuntimeError(
            "Non-finite prediction probabilities."
        )

    if np.any(probabilities < -1e-6):
        raise RuntimeError(
            "Invalid negative prediction probabilities."
        )

    if not np.allclose(
        probabilities.sum(axis=1),
        1.0,
        atol=1e-4,
        rtol=0
    ):
        raise RuntimeError(
            "Prediction probabilities do not sum to one."
        )

    prediction_indices[offset:end] = (
        np.argmax(probabilities, axis=1)
    )

inference_seconds = (
    time.perf_counter() - prediction_start
)

if np.any(prediction_indices < 0) or np.any(
    prediction_indices >= 5
):
    raise RuntimeError(
        "Predicted class indices outside 0–4."
    )

NSL10_TUNED_VAL_TRUE = y_true
NSL10_TUNED_VAL_PRED = prediction_indices

labels = list(range(5))

metrics = {
    "Accuracy": accuracy_score(
        y_true, prediction_indices
    ),
    "Macro Precision": precision_score(
        y_true, prediction_indices,
        labels=labels,
        average="macro",
        zero_division=0
    ),
    "Macro Recall": recall_score(
        y_true, prediction_indices,
        labels=labels,
        average="macro",
        zero_division=0
    ),
    "Macro F1": f1_score(
        y_true, prediction_indices,
        labels=labels,
        average="macro",
        zero_division=0
    ),
    "Weighted F1": f1_score(
        y_true, prediction_indices,
        labels=labels,
        average="weighted",
        zero_division=0
    ),
    "Balanced Accuracy": balanced_accuracy_score(
        y_true, prediction_indices
    ),
    "MCC": matthews_corrcoef(
        y_true, prediction_indices
    )
}

report_text = classification_report(
    y_true,
    prediction_indices,
    labels=labels,
    target_names=NSL10_CLASS_NAMES,
    digits=6,
    zero_division=0
)

report_dict = classification_report(
    y_true,
    prediction_indices,
    labels=labels,
    target_names=NSL10_CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(
    report_dict
).transpose()

cm = confusion_matrix(
    y_true,
    prediction_indices,
    labels=labels
)

cm_df = pd.DataFrame(
    cm,
    index=NSL10_CLASS_NAMES,
    columns=NSL10_CLASS_NAMES
)

if cm.shape != (5, 5) or cm.sum() != N:
    raise RuntimeError(
        "Validation confusion matrix verification failed."
    )

if int(np.trace(cm)) != int(
    np.count_nonzero(prediction_indices == y_true)
):
    raise RuntimeError(
        "Confusion matrix correct-count mismatch."
    )

if not np.isclose(
    metrics["Macro F1"],
    float(report_df.loc["macro avg", "f1-score"]),
    rtol=0,
    atol=1e-12
):
    raise RuntimeError(
        "Macro F1 consistency verification failed."
    )

# Export validation results without overwriting files.

metrics_record = {
    "Model": MODEL_NAME,
    "Partition": "validation",
    "Records": N,
    **metrics
}

metrics_bytes = pd.DataFrame(
    [metrics_record]
).to_csv(
    index=False,
    float_format="%.9f"
).encode("utf-8")

report_bytes = report_df.to_csv(
    index_label="Class",
    float_format="%.9f"
).encode("utf-8")

cm_bytes = cm_df.to_csv(
    index_label="Actual Class"
).encode("utf-8")

prediction_buffer = io.BytesIO()

np.save(
    prediction_buffer,
    prediction_indices.astype(np.int8),
    allow_pickle=False
)

exports = {
    "RF_H2_NSL_validation_metrics.csv":
        metrics_bytes,
    "RF_H2_NSL_validation_classification_report.csv":
        report_bytes,
    "RF_H2_NSL_validation_confusion_matrix.csv":
        cm_bytes,
    "RF_H2_NSL_validation_predictions.npy":
        prediction_buffer.getvalue()
}

print("\nRESULT EXPORTS")

for filename, content in exports.items():

    status = ns10_write_once(
        RESULT_DIR / filename,
        content
    )

    print(f"{filename}: {status}")

NSL10_TUNED_VALIDATION_METRICS = metrics.copy()
NSL10_TUNED_VALIDATED = True

print("\nVALIDATION METRICS")

for name, value in metrics.items():
    print(f"{name:<22}: {value:.6f}")

print("\nCLASSIFICATION REPORT")
print(report_text)

print("CONFUSION MATRIX")
print(cm_df.to_string())

print("\nEVALUATION SUMMARY")
print("Model:", MODEL_NAME)
print("SavedModel reloaded: PASS")
print("Serving input features:", len(input_signature))
print("Prediction observations:", N)
print("Class-index range: 0–4")
print("Classification report: PASS")
print("Confusion matrix: PASS")
print("Validation result exports:", len(exports))
print(
    "Inference time (seconds):",
    round(inference_seconds, 3)
)

print("\nCELL 08: PASSED")
print("Frozen-tuning validation evaluation: COMPLETE")
print("Official test evaluation: NOT PERFORMED")
print("=" * 72)


CELL 08 — FROZEN-TUNING VALIDATION EVALUATION

RESULT EXPORTS
RF_H2_NSL_validation_metrics.csv: CREATED
RF_H2_NSL_validation_classification_report.csv: CREATED
RF_H2_NSL_validation_confusion_matrix.csv: CREATED
RF_H2_NSL_validation_predictions.npy: CREATED

VALIDATION METRICS
Accuracy              : 0.999312
Macro Precision       : 0.971182
Macro Recall          : 0.946560
Macro F1              : 0.958147
Weighted F1           : 0.999306
Balanced Accuracy     : 0.946560
MCC                   : 0.998799

CLASSIFICATION REPORT
              precision    recall  f1-score   support

      Normal   0.998912  0.999802  0.999357     10102
         DoS   0.999855  0.999855  0.999855      6889
       Probe   1.000000  0.996568  0.998281      1748
         R2L   1.000000  0.986577  0.993243       149
         U2R   0.857143  0.750000  0.800000         8

    accuracy                       0.999312     18896
   macro avg   0.971182  0.946560  0.958147     18896
weighted avg   0.999305  0.999312  

In [ ]:

# ============================================================
# NOTEBOOK 10 — NSL-KDD RANDOM FOREST REPLICATION
# CELL 09 — VALIDATION COMPARISON AND CONFIGURATION AUDIT
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("=" * 72)
print("CELL 09 — VALIDATION COMPARISON AND AUDIT")
print("=" * 72)

if not globals().get("NSL10_BASELINE_VALIDATED", False):
    raise RuntimeError("Cell 06 must pass first.")

if not globals().get("NSL10_TUNED_VALIDATED", False):
    raise RuntimeError("Cell 08 must pass first.")

NSL10_VALIDATION_COMPARISON_VERIFIED = False

NSL10_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

model_sources = {
    "RF-B-NSL": {
        "directory": NSL10_MODEL_RESULT_DIRS["RF-B-NSL"],
        "prefix": "RF_B_NSL"
    },
    "RF-H2-NSL": {
        "directory": NSL10_MODEL_RESULT_DIRS["RF-H2-NSL"],
        "prefix": "RF_H2_NSL"
    }
}

y_true = NSL10_PARTITIONS["validation"][
    "Label"
].to_numpy(dtype=np.int32)

if len(y_true) != 18896:
    raise RuntimeError(
        "Unexpected validation observation count."
    )

metric_names = [
    "Accuracy",
    "Macro Precision",
    "Macro Recall",
    "Macro F1",
    "Weighted F1",
    "Balanced Accuracy",
    "MCC"
]

model_metrics = {}
model_reports = {}
model_predictions = {}
model_matrices = {}

for model_name, source in model_sources.items():

    directory = source["directory"]
    prefix = source["prefix"]

    metrics_path = (
        directory / f"{prefix}_validation_metrics.csv"
    )

    report_path = (
        directory
        / f"{prefix}_validation_classification_report.csv"
    )

    matrix_path = (
        directory
        / f"{prefix}_validation_confusion_matrix.csv"
    )

    prediction_path = (
        directory / f"{prefix}_validation_predictions.npy"
    )

    for path in [
        metrics_path,
        report_path,
        matrix_path,
        prediction_path
    ]:
        if not path.is_file():
            raise FileNotFoundError(path)

    metrics_df = pd.read_csv(metrics_path)

    if len(metrics_df) != 1:
        raise RuntimeError(
            f"{model_name}: expected one metrics record."
        )

    metrics_row = metrics_df.iloc[0]

    if metrics_row["Model"] != model_name:
        raise RuntimeError(
            f"{model_name}: model identifier mismatch."
        )

    if metrics_row["Partition"] != "validation":
        raise RuntimeError(
            f"{model_name}: partition identifier mismatch."
        )

    if int(metrics_row["Records"]) != len(y_true):
        raise RuntimeError(
            f"{model_name}: validation count mismatch."
        )

    predictions = np.load(
        prediction_path,
        allow_pickle=False
    ).astype(np.int32)

    if predictions.shape != y_true.shape:
        raise RuntimeError(
            f"{model_name}: prediction length mismatch."
        )

    if np.any(predictions < 0) or np.any(predictions >= 5):
        raise RuntimeError(
            f"{model_name}: invalid prediction indices."
        )

    calculated_accuracy = accuracy_score(
        y_true,
        predictions
    )

    calculated_macro_f1 = f1_score(
        y_true,
        predictions,
        labels=list(range(5)),
        average="macro",
        zero_division=0
    )

    for metric_name, actual_value in [
        ("Accuracy", calculated_accuracy),
        ("Macro F1", calculated_macro_f1)
    ]:
        if not np.isclose(
            actual_value,
            float(metrics_row[metric_name]),
            rtol=0,
            atol=2e-9
        ):
            raise RuntimeError(
                f"{model_name}: {metric_name} does not "
                "match saved predictions."
            )

    report_df = pd.read_csv(
        report_path,
        index_col=0
    )

    if not set(NSL10_CLASS_NAMES).issubset(
        report_df.index
    ):
        raise RuntimeError(
            f"{model_name}: class report mismatch."
        )

    computed_report = classification_report(
        y_true,
        predictions,
        labels=list(range(5)),
        target_names=NSL10_CLASS_NAMES,
        output_dict=True,
        zero_division=0
    )

    for class_name in NSL10_CLASS_NAMES:

        for metric in [
            "precision", "recall", "f1-score", "support"
        ]:
            archived = float(
                report_df.loc[class_name, metric]
            )

            calculated = float(
                computed_report[class_name][metric]
            )

            if not np.isclose(
                archived,
                calculated,
                rtol=0,
                atol=2e-9
            ):
                raise RuntimeError(
                    f"{model_name}: class report mismatch "
                    f"for {class_name}, {metric}."
                )

    saved_cm = pd.read_csv(
        matrix_path,
        index_col=0
    )

    if (
        list(saved_cm.index) != NSL10_CLASS_NAMES
        or list(saved_cm.columns) != NSL10_CLASS_NAMES
    ):
        raise RuntimeError(
            f"{model_name}: confusion matrix class order mismatch."
        )

    computed_cm = confusion_matrix(
        y_true,
        predictions,
        labels=list(range(5))
    )

    if not np.array_equal(
        saved_cm.to_numpy(dtype=np.int64),
        computed_cm
    ):
        raise RuntimeError(
            f"{model_name}: confusion matrix mismatch."
        )

    model_metrics[model_name] = metrics_row
    model_reports[model_name] = report_df
    model_predictions[model_name] = predictions
    model_matrices[model_name] = computed_cm

    print(f"{model_name}: SAVED VALIDATION RESULTS VERIFIED")

# Verify the same validation targets for both models.

if not np.array_equal(
    NSL10_BASELINE_VAL_TRUE,
    NSL10_TUNED_VAL_TRUE
):
    raise RuntimeError(
        "Validation target alignment mismatch."
    )

if not np.array_equal(
    model_predictions["RF-B-NSL"],
    NSL10_BASELINE_VAL_PRED
):
    raise RuntimeError(
        "Baseline prediction alignment mismatch."
    )

if not np.array_equal(
    model_predictions["RF-H2-NSL"],
    NSL10_TUNED_VAL_PRED
):
    raise RuntimeError(
        "Frozen-tuning prediction alignment mismatch."
    )

# Compare the aggregate validation metrics.

comparison_records = []

for metric in metric_names:

    baseline_value = float(
        model_metrics["RF-B-NSL"][metric]
    )

    tuned_value = float(
        model_metrics["RF-H2-NSL"][metric]
    )

    comparison_records.append({
        "Metric": metric,
        "RF-B-NSL": baseline_value,
        "RF-H2-NSL": tuned_value,
        "Change": tuned_value - baseline_value
    })

NSL10_VALIDATION_COMPARISON = pd.DataFrame(
    comparison_records
)

print("\nAGGREGATE VALIDATION COMPARISON")
print(
    NSL10_VALIDATION_COMPARISON.to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}"
    )
)

# Compare class-level validation performance.

class_records = []

baseline_report = model_reports["RF-B-NSL"]
tuned_report = model_reports["RF-H2-NSL"]

for class_name in NSL10_CLASS_NAMES:

    baseline_f1 = float(
        baseline_report.loc[class_name, "f1-score"]
    )

    tuned_f1 = float(
        tuned_report.loc[class_name, "f1-score"]
    )

    baseline_recall = float(
        baseline_report.loc[class_name, "recall"]
    )

    tuned_recall = float(
        tuned_report.loc[class_name, "recall"]
    )

    baseline_precision = float(
        baseline_report.loc[class_name, "precision"]
    )

    tuned_precision = float(
        tuned_report.loc[class_name, "precision"]
    )

    baseline_support = int(
        baseline_report.loc[class_name, "support"]
    )

    tuned_support = int(
        tuned_report.loc[class_name, "support"]
    )

    if baseline_support != tuned_support:
        raise RuntimeError(
            f"Class support mismatch: {class_name}"
        )

    class_records.append({
        "Class": class_name,
        "Support": baseline_support,
        "Baseline Precision": baseline_precision,
        "Tuned Precision": tuned_precision,
        "Precision Change": (
            tuned_precision - baseline_precision
        ),
        "Baseline Recall": baseline_recall,
        "Tuned Recall": tuned_recall,
        "Recall Change": (
            tuned_recall - baseline_recall
        ),
        "Baseline F1": baseline_f1,
        "Tuned F1": tuned_f1,
        "F1 Change": tuned_f1 - baseline_f1
    })

NSL10_VALIDATION_CLASS_COMPARISON = pd.DataFrame(
    class_records
)

if int(
    NSL10_VALIDATION_CLASS_COMPARISON["Support"].sum()
) != 18896:
    raise RuntimeError(
        "Per-class validation support mismatch."
    )

print("\nPER-CLASS VALIDATION COMPARISON")
print(
    NSL10_VALIDATION_CLASS_COMPARISON[
        [
            "Class",
            "Support",
            "Baseline F1",
            "Tuned F1",
            "F1 Change",
            "Recall Change"
        ]
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}"
    )
)

# Verify that experimental configurations remain frozen.

expected_configs = {
    "RF-B-NSL": {
        "num_trees": 100,
        "max_depth": -1,
        "min_examples": 1,
        "random_seed": 42
    },
    "RF-H2-NSL": {
        "num_trees": 100,
        "max_depth": 30,
        "min_examples": 1,
        "random_seed": 42
    }
}

if NSL10_MODEL_CONFIGS != expected_configs:
    raise RuntimeError(
        "Experiment configurations have changed."
    )

experiment_record = {
    "dataset": "NSL-KDD",
    "stage": "independent_random_forest_replication",
    "training_partition": "development_train",
    "training_observations": 107077,
    "validation_partition": "development_validation",
    "validation_observations": 18896,
    "official_test_observations": 22544,
    "encoded_predictors": 122,
    "class_order": NSL10_CLASS_NAMES,
    "model_configurations": NSL10_MODEL_CONFIGS,
    "training_times_seconds": {
        "RF-B-NSL": NSL10_BASELINE_TRAIN_SECONDS,
        "RF-H2-NSL": NSL10_TUNED_TRAIN_SECONDS
    },
    "source_preprocessing_fingerprint": (
        NSL10_EXPECTED_FINGERPRINT
    ),
    "hyperparameter_search_performed": False,
    "validation_derived_configuration_changes": False,
    "resampling_performed": False,
    "cic_model_transfer_performed": False,
    "official_test_evaluation_performed": False
}

# Export verified validation comparison.

comparison_bytes = (
    NSL10_VALIDATION_COMPARISON.to_csv(
        index=False,
        float_format="%.9f"
    ).encode("utf-8")
)

class_comparison_bytes = (
    NSL10_VALIDATION_CLASS_COMPARISON.to_csv(
        index=False,
        float_format="%.9f"
    ).encode("utf-8")
)

experiment_bytes = (
    json.dumps(
        experiment_record,
        indent=2,
        sort_keys=True,
        allow_nan=False
    ) + "\n"
).encode("utf-8")

print("\nRESULT EXPORTS")

exports = {
    "NSL10_validation_model_comparison.csv":
        comparison_bytes,
    "NSL10_validation_class_comparison.csv":
        class_comparison_bytes,
    "NSL10_locked_experiment_configuration.json":
        experiment_bytes
}

for filename, content in exports.items():

    status = ns10_write_once(
        NSL10_RESULTS_DIR / filename,
        content
    )

    print(f"{filename}: {status}")

NSL10_VALIDATION_COMPARISON_VERIFIED = True

print("\nCONFIGURATION AUDIT")
print("Baseline configuration: LOCKED")
print("Frozen-tuning configuration: LOCKED")
print("Hyperparameter search performed: NO")
print("Validation-derived configuration changes: NO")
print("Resampling performed: NO")
print("CIC model transfer performed: NO")
print("Official test evaluation: PENDING")
print("Validation results aligned: PASS")

print("\nCELL 09: PASSED")
print("Both NSL validation evaluations: COMPLETE")
print("Official test evaluation: NOT PERFORMED")
print("=" * 72)


CELL 09 — VALIDATION COMPARISON AND AUDIT
RF-B-NSL: SAVED VALIDATION RESULTS VERIFIED
RF-H2-NSL: SAVED VALIDATION RESULTS VERIFIED

AGGREGATE VALIDATION COMPARISON
           Metric  RF-B-NSL  RF-H2-NSL   Change
         Accuracy  0.999312   0.999312 0.000000
  Macro Precision  0.971182   0.971182 0.000000
     Macro Recall  0.946560   0.946560 0.000000
         Macro F1  0.958147   0.958147 0.000000
      Weighted F1  0.999306   0.999306 0.000000
Balanced Accuracy  0.946560   0.946560 0.000000
              MCC  0.998799   0.998799 0.000000

PER-CLASS VALIDATION COMPARISON
 Class  Support  Baseline F1  Tuned F1  F1 Change  Recall Change
Normal    10102     0.999357  0.999357   0.000000       0.000000
   DoS     6889     0.999855  0.999855   0.000000       0.000000
 Probe     1748     0.998281  0.998281   0.000000       0.000000
   R2L      149     0.993243  0.993243   0.000000       0.000000
   U2R        8     0.800000  0.800000   0.000000       0.000000

RESULT EXPORTS
NSL10_validat

In [ ]:

# ============================================================
# NOTEBOOK 10 — NSL-KDD RANDOM FOREST REPLICATION
# CELL 10 — BASELINE OFFICIAL TEST EVALUATION
# ============================================================

from pathlib import Path
import io
import hashlib
import time

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    classification_report,
    confusion_matrix
)

print("=" * 72)
print("CELL 10 — BASELINE OFFICIAL TEST EVALUATION")
print("=" * 72)

if not globals().get(
    "NSL10_VALIDATION_COMPARISON_VERIFIED", False
):
    raise RuntimeError("Cell 09 must pass first.")

NSL10_BASELINE_TEST_VERIFIED = False

NSL10_TEST_FRAME = NSL10_PARTITIONS["test"]
NSL10_TEST_Y = NSL10_TEST_FRAME[
    "Label"
].to_numpy(dtype=np.int32)

NSL10_TEST_N = len(NSL10_TEST_Y)

NSL10_LABELS = list(range(5))

if NSL10_TEST_N != 22544:
    raise RuntimeError("Unexpected official test size.")

if not np.array_equal(
    np.bincount(NSL10_TEST_Y, minlength=5),
    np.array([9711, 7460, 2421, 2885, 67])
):
    raise RuntimeError(
        "Official test class distribution mismatch."
    )

def ns10_test_write_once(path, content):

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    try:
        with path.open("xb") as handle:
            handle.write(content)
        return "CREATED"

    except FileExistsError:
        if hashlib.sha256(path.read_bytes()).digest() == (
            hashlib.sha256(content).digest()
        ):
            return "ALREADY VERIFIED"

        raise RuntimeError(
            f"Existing artefact differs: {path}"
        )

def ns10_predict_saved_signature(
    signature, frame, batch_size=4096
):

    _, input_specs = (
        signature.structured_input_signature
    )

    if set(input_specs) != set(NSL10_FEATURE_NAMES):
        raise RuntimeError(
            "SavedModel input feature names differ."
        )

    if len(input_specs) != 122:
        raise RuntimeError(
            "SavedModel input feature count differs."
        )

    for feature_name, spec in input_specs.items():
        if spec.dtype != tf.float32:
            raise RuntimeError(
                f"Unexpected input dtype: {feature_name}"
            )

    n = len(frame)
    predictions = np.empty(n, dtype=np.int8)

    for offset in range(0, n, batch_size):

        end = min(offset + batch_size, n)

        batch = frame.iloc[offset:end]

        model_inputs = {
            feature_name: tf.convert_to_tensor(
                batch[feature_name].to_numpy(
                    dtype=np.float32
                ),
                dtype=tf.float32
            )
            for feature_name in input_specs
        }

        outputs = signature(**model_inputs)

        if len(outputs) != 1:
            raise RuntimeError(
                "Unexpected SavedModel output count."
            )

        probabilities = next(
            iter(outputs.values())
        ).numpy()

        if probabilities.shape != (end - offset, 5):
            raise RuntimeError(
                f"Invalid prediction shape: "
                f"{probabilities.shape}"
            )

        if not np.isfinite(probabilities).all():
            raise RuntimeError(
                "Non-finite prediction scores."
            )

        if np.any(probabilities < -1e-6):
            raise RuntimeError(
                "Invalid negative prediction probabilities."
            )

        if not np.allclose(
            probabilities.sum(axis=1),
            1.0,
            rtol=0,
            atol=1e-4
        ):
            raise RuntimeError(
                "Prediction probabilities do not sum to one."
            )

        predictions[offset:end] = np.argmax(
            probabilities, axis=1
        ).astype(np.int8)

    if np.any(predictions < 0) or np.any(predictions > 4):
        raise RuntimeError("Invalid predicted class index.")

    return predictions

def ns10_calculate_test_metrics(y_true, y_pred):

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro Precision": precision_score(
            y_true, y_pred,
            labels=NSL10_LABELS,
            average="macro",
            zero_division=0
        ),
        "Macro Recall": recall_score(
            y_true, y_pred,
            labels=NSL10_LABELS,
            average="macro",
            zero_division=0
        ),
        "Macro F1": f1_score(
            y_true, y_pred,
            labels=NSL10_LABELS,
            average="macro",
            zero_division=0
        ),
        "Weighted F1": f1_score(
            y_true, y_pred,
            labels=NSL10_LABELS,
            average="weighted",
            zero_division=0
        ),
        "Balanced Accuracy": balanced_accuracy_score(
            y_true, y_pred
        ),
        "MCC": matthews_corrcoef(
            y_true, y_pred
        )
    }

def ns10_evaluate_official_test(
    model_name, prefix
):

    model_path = NSL10_MODEL_PATHS[model_name]
    result_dir = NSL10_MODEL_RESULT_DIRS[model_name]

    if not (model_path / "saved_model.pb").is_file():
        raise FileNotFoundError(
            model_path / "saved_model.pb"
        )

    saved_model = tf.saved_model.load(
        str(model_path)
    )

    if "serving_default" not in saved_model.signatures:
        raise RuntimeError(
            "SavedModel serving signature unavailable."
        )

    signature = saved_model.signatures[
        "serving_default"
    ]

    # Check prediction-index alignment against
    # the saved validation predictions.

    validation_frame = NSL10_PARTITIONS["validation"]

    validation_targets = validation_frame[
        "Label"
    ].to_numpy(dtype=np.int32)

    selected_positions = []

    for class_index in NSL10_LABELS:

        class_positions = np.flatnonzero(
            validation_targets == class_index
        )

        selected_positions.extend(
            class_positions[:8].tolist()
        )

    selected_positions = np.asarray(
        sorted(set(selected_positions)),
        dtype=np.int64
    )

    sampled_frame = validation_frame.iloc[
        selected_positions
    ]

    sampled_predictions = ns10_predict_saved_signature(
        signature,
        sampled_frame
    )

    saved_validation_predictions = np.load(
        result_dir / f"{prefix}_validation_predictions.npy",
        allow_pickle=False
    )

    if not np.array_equal(
        sampled_predictions,
        saved_validation_predictions[selected_positions]
    ):
        raise RuntimeError(
            f"{model_name}: saved validation prediction "
            "alignment failed."
        )

    # Evaluate the official test partition.

    inference_start = time.perf_counter()

    predictions = ns10_predict_saved_signature(
        signature,
        NSL10_TEST_FRAME
    )

    inference_seconds = (
        time.perf_counter() - inference_start
    )

    metrics = ns10_calculate_test_metrics(
        NSL10_TEST_Y,
        predictions
    )

    report_text = classification_report(
        NSL10_TEST_Y,
        predictions,
        labels=NSL10_LABELS,
        target_names=NSL10_CLASS_NAMES,
        digits=6,
        zero_division=0
    )

    report_dict = classification_report(
        NSL10_TEST_Y,
        predictions,
        labels=NSL10_LABELS,
        target_names=NSL10_CLASS_NAMES,
        output_dict=True,
        zero_division=0
    )

    report_df = pd.DataFrame(
        report_dict
    ).transpose()

    matrix = confusion_matrix(
        NSL10_TEST_Y,
        predictions,
        labels=NSL10_LABELS
    )

    if matrix.shape != (5, 5):
        raise RuntimeError(
            "Unexpected test confusion matrix shape."
        )

    if matrix.sum() != NSL10_TEST_N:
        raise RuntimeError(
            "Official test confusion matrix total mismatch."
        )

    if np.trace(matrix) != np.count_nonzero(
        NSL10_TEST_Y == predictions
    ):
        raise RuntimeError(
            "Confusion matrix correct-count mismatch."
        )

    if not np.isclose(
        metrics["Macro F1"],
        float(
            report_df.loc["macro avg", "f1-score"]
        ),
        atol=1e-12,
        rtol=0
    ):
        raise RuntimeError(
            "Classification report Macro F1 mismatch."
        )

    matrix_df = pd.DataFrame(
        matrix,
        index=NSL10_CLASS_NAMES,
        columns=NSL10_CLASS_NAMES
    )

    metrics_record = {
        "Model": model_name,
        "Partition": "official_test",
        "Records": NSL10_TEST_N,
        **metrics
    }

    metrics_content = pd.DataFrame(
        [metrics_record]
    ).to_csv(
        index=False,
        float_format="%.9f"
    ).encode("utf-8")

    report_content = report_df.to_csv(
        index_label="Class",
        float_format="%.9f"
    ).encode("utf-8")

    matrix_content = matrix_df.to_csv(
        index_label="Actual Class"
    ).encode("utf-8")

    prediction_buffer = io.BytesIO()

    np.save(
        prediction_buffer,
        predictions,
        allow_pickle=False
    )

    exports = {
        f"{prefix}_test_metrics.csv":
            metrics_content,
        f"{prefix}_test_classification_report.csv":
            report_content,
        f"{prefix}_test_confusion_matrix.csv":
            matrix_content,
        f"{prefix}_test_predictions.npy":
            prediction_buffer.getvalue()
    }

    print("\nRESULT EXPORTS")

    for filename, content in exports.items():

        status = ns10_test_write_once(
            result_dir / filename,
            content
        )

        print(f"{filename}: {status}")

    print("\nOFFICIAL TEST METRICS")

    for metric, value in metrics.items():
        print(f"{metric:<22}: {value:.6f}")

    print("\nCLASSIFICATION REPORT")
    print(report_text)

    print("CONFUSION MATRIX")
    print(matrix_df.to_string())

    print("\nEVALUATION SUMMARY")
    print("Model:", model_name)
    print("SavedModel reloaded: PASS")
    print(
        "Validation sample alignment: PASS",
        f"({len(selected_positions)} observations)"
    )
    print("Official test observations:", NSL10_TEST_N)
    print("Input features:", len(NSL10_FEATURE_NAMES))
    print("Output classes:", len(NSL10_CLASS_NAMES))
    print(
        "Correct classifications:",
        int(np.count_nonzero(
            NSL10_TEST_Y == predictions
        ))
    )
    print(
        "Inference time (seconds):",
        round(inference_seconds, 3)
    )
    print("Classification report: PASS")
    print("Confusion matrix: PASS")
    print("Result exports:", len(exports))

    return {
        "metrics": metrics,
        "predictions": predictions,
        "confusion_matrix": matrix,
        "report": report_df,
        "inference_seconds": inference_seconds
    }

NSL10_BASELINE_TEST = ns10_evaluate_official_test(
    "RF-B-NSL",
    "RF_B_NSL"
)

NSL10_BASELINE_TEST_VERIFIED = True

print("\nCELL 10: PASSED")
print("Baseline official test evaluation: COMPLETE")
print("Model training performed: NO")
print("=" * 72)


CELL 10 — BASELINE OFFICIAL TEST EVALUATION

RESULT EXPORTS
RF_B_NSL_test_metrics.csv: CREATED
RF_B_NSL_test_classification_report.csv: CREATED
RF_B_NSL_test_confusion_matrix.csv: CREATED
RF_B_NSL_test_predictions.npy: CREATED

OFFICIAL TEST METRICS
Accuracy              : 0.758384
Macro Precision       : 0.766484
Macro Recall          : 0.493121
Macro F1              : 0.504649
Weighted F1           : 0.717539
Balanced Accuracy     : 0.493121
MCC                   : 0.650888

CLASSIFICATION REPORT
              precision    recall  f1-score   support

      Normal   0.656561  0.972299  0.783829      9711
         DoS   0.963205  0.810590  0.880332      7460
       Probe   0.843331  0.598100  0.699855      2421
         R2L   0.969325  0.054766  0.103675      2885
         U2R   0.400000  0.029851  0.055556        67

    accuracy                       0.758384     22544
   macro avg   0.766484  0.493121  0.504649     22544
weighted avg   0.817352  0.758384  0.717539     22544

CONFUSI

In [ ]:

# ============================================================
# NOTEBOOK 10 — NSL-KDD RANDOM FOREST REPLICATION
# CELL 11 — FROZEN-TUNING OFFICIAL TEST EVALUATION
# ============================================================

import numpy as np

print("=" * 72)
print("CELL 11 — FROZEN-TUNING OFFICIAL TEST EVALUATION")
print("=" * 72)

if not globals().get(
    "NSL10_BASELINE_TEST_VERIFIED", False
):
    raise RuntimeError("Cell 10 must pass first.")

if not globals().get(
    "NSL10_TUNED_VALIDATED", False
):
    raise RuntimeError("Cell 08 must pass first.")

NSL10_TUNED_TEST_VERIFIED = False

expected_configuration = {
    "num_trees": 100,
    "max_depth": 30,
    "min_examples": 1,
    "random_seed": 42
}

if (
    NSL10_MODEL_CONFIGS["RF-H2-NSL"]
    != expected_configuration
):
    raise RuntimeError(
        "Frozen-tuning configuration has changed."
    )

NSL10_TUNED_TEST = ns10_evaluate_official_test(
    "RF-H2-NSL",
    "RF_H2_NSL"
)

# Verify identical test partition and prediction dimensions.

baseline_predictions = (
    NSL10_BASELINE_TEST["predictions"]
)

tuned_predictions = (
    NSL10_TUNED_TEST["predictions"]
)

if not (
    baseline_predictions.shape
    == tuned_predictions.shape
    == NSL10_TEST_Y.shape
    == (22544,)
):
    raise RuntimeError(
        "Official test prediction alignment failed."
    )

if not np.array_equal(
    np.bincount(
        NSL10_TEST_Y,
        minlength=5
    ),
    [9711, 7460, 2421, 2885, 67]
):
    raise RuntimeError(
        "Official test targets changed."
    )

NSL10_TUNED_TEST_VERIFIED = True

print("\nPAIRED TEST READINESS")
print(
    "Baseline predictions:",
    len(baseline_predictions)
)
print(
    "Frozen-tuning predictions:",
    len(tuned_predictions)
)
print("True labels:", len(NSL10_TEST_Y))
print("Prediction alignment: PASS")

print("\nCELL 11: PASSED")
print("Frozen-tuning official test evaluation: COMPLETE")
print("Both models evaluated on identical test records: YES")
print("Model training performed: NO")
print("=" * 72)


CELL 11 — FROZEN-TUNING OFFICIAL TEST EVALUATION

RESULT EXPORTS
RF_H2_NSL_test_metrics.csv: CREATED
RF_H2_NSL_test_classification_report.csv: CREATED
RF_H2_NSL_test_confusion_matrix.csv: CREATED
RF_H2_NSL_test_predictions.npy: CREATED

OFFICIAL TEST METRICS
Accuracy              : 0.758384
Macro Precision       : 0.766484
Macro Recall          : 0.493121
Macro F1              : 0.504649
Weighted F1           : 0.717539
Balanced Accuracy     : 0.493121
MCC                   : 0.650888

CLASSIFICATION REPORT
              precision    recall  f1-score   support

      Normal   0.656561  0.972299  0.783829      9711
         DoS   0.963205  0.810590  0.880332      7460
       Probe   0.843331  0.598100  0.699855      2421
         R2L   0.969325  0.054766  0.103675      2885
         U2R   0.400000  0.029851  0.055556        67

    accuracy                       0.758384     22544
   macro avg   0.766484  0.493121  0.504649     22544
weighted avg   0.817352  0.758384  0.717539     22544

In [ ]:

# ============================================================
# NOTEBOOK 10 — NSL-KDD RANDOM FOREST REPLICATION
# CELL 12 — FINAL EVALUATION AND COMPLETION AUDIT
# ============================================================

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd

from sklearn.metrics import (
    confusion_matrix,
    classification_report
)

print("=" * 72)
print("CELL 12 — FINAL EVALUATION AND COMPLETION AUDIT")
print("=" * 72)

if not globals().get(
    "NSL10_BASELINE_TEST_VERIFIED", False
):
    raise RuntimeError("Cell 10 must pass first.")

if not globals().get(
    "NSL10_TUNED_TEST_VERIFIED", False
):
    raise RuntimeError("Cell 11 must pass first.")

NSL10_NOTEBOOK_COMPLETE = False

model_specifications = {
    "RF-B-NSL": "RF_B_NSL",
    "RF-H2-NSL": "RF_H2_NSL"
}

metric_names = [
    "Accuracy",
    "Macro Precision",
    "Macro Recall",
    "Macro F1",
    "Weighted F1",
    "Balanced Accuracy",
    "MCC"
]

verified_test_metrics = {}
verified_test_reports = {}
verified_test_predictions = {}

print("\nSAVED OFFICIAL TEST RESULT VERIFICATION")

for model_name, prefix in model_specifications.items():

    result_dir = NSL10_MODEL_RESULT_DIRS[model_name]

    metrics_path = (
        result_dir / f"{prefix}_test_metrics.csv"
    )

    report_path = (
        result_dir
        / f"{prefix}_test_classification_report.csv"
    )

    matrix_path = (
        result_dir / f"{prefix}_test_confusion_matrix.csv"
    )

    predictions_path = (
        result_dir / f"{prefix}_test_predictions.npy"
    )

    for path in [
        metrics_path,
        report_path,
        matrix_path,
        predictions_path
    ]:
        if not path.is_file():
            raise FileNotFoundError(path)

    predictions = np.load(
        predictions_path,
        allow_pickle=False
    ).astype(np.int32)

    if predictions.shape != (22544,):
        raise RuntimeError(
            f"{model_name}: invalid prediction shape."
        )

    if np.any(predictions < 0) or np.any(predictions > 4):
        raise RuntimeError(
            f"{model_name}: invalid class indices."
        )

    metrics_frame = pd.read_csv(metrics_path)

    if len(metrics_frame) != 1:
        raise RuntimeError(
            f"{model_name}: invalid metric record count."
        )

    saved_metrics = metrics_frame.iloc[0]

    if saved_metrics["Model"] != model_name:
        raise RuntimeError(
            f"{model_name}: model identifier mismatch."
        )

    if saved_metrics["Partition"] != "official_test":
        raise RuntimeError(
            f"{model_name}: partition identifier mismatch."
        )

    if int(saved_metrics["Records"]) != 22544:
        raise RuntimeError(
            f"{model_name}: test observation count mismatch."
        )

    recomputed = ns10_calculate_test_metrics(
        NSL10_TEST_Y,
        predictions
    )

    for metric in metric_names:

        if not np.isclose(
            float(saved_metrics[metric]),
            float(recomputed[metric]),
            rtol=0,
            atol=2e-9
        ):
            raise RuntimeError(
                f"{model_name}: saved {metric} mismatch."
            )

    saved_report = pd.read_csv(
        report_path,
        index_col=0
    )

    recomputed_report = classification_report(
        NSL10_TEST_Y,
        predictions,
        labels=NSL10_LABELS,
        target_names=NSL10_CLASS_NAMES,
        output_dict=True,
        zero_division=0
    )

    for class_name in NSL10_CLASS_NAMES:

        for field in [
            "precision",
            "recall",
            "f1-score",
            "support"
        ]:

            actual = float(
                recomputed_report[class_name][field]
            )

            archived = float(
                saved_report.loc[class_name, field]
            )

            if not np.isclose(
                actual,
                archived,
                rtol=0,
                atol=2e-9
            ):
                raise RuntimeError(
                    f"{model_name}: report mismatch "
                    f"for {class_name}, {field}."
                )

    saved_matrix = pd.read_csv(
        matrix_path,
        index_col=0
    )

    if (
        list(saved_matrix.index) != NSL10_CLASS_NAMES
        or list(saved_matrix.columns) != NSL10_CLASS_NAMES
    ):
        raise RuntimeError(
            f"{model_name}: confusion matrix labels differ."
        )

    recomputed_matrix = confusion_matrix(
        NSL10_TEST_Y,
        predictions,
        labels=NSL10_LABELS
    )

    if not np.array_equal(
        saved_matrix.to_numpy(dtype=np.int64),
        recomputed_matrix
    ):
        raise RuntimeError(
            f"{model_name}: confusion matrix mismatch."
        )

    verified_test_metrics[model_name] = recomputed
    verified_test_reports[model_name] = saved_report
    verified_test_predictions[model_name] = predictions

    print(f"{model_name}: PASS")

# Verify original locked experiment configuration.

locked_configuration_path = (
    NSL10_RESULTS_DIR
    / "NSL10_locked_experiment_configuration.json"
)

locked_record = json.loads(
    locked_configuration_path.read_text(
        encoding="utf-8"
    )
)

if locked_record["model_configurations"] != (
    NSL10_MODEL_CONFIGS
):
    raise RuntimeError(
        "Locked experiment configuration mismatch."
    )

if locked_record[
    "source_preprocessing_fingerprint"
] != NSL10_EXPECTED_FINGERPRINT:
    raise RuntimeError(
        "Preprocessing fingerprint mismatch."
    )

for key in [
    "hyperparameter_search_performed",
    "validation_derived_configuration_changes",
    "resampling_performed",
    "cic_model_transfer_performed"
]:
    if locked_record[key] is not False:
        raise RuntimeError(
            f"Unexpected experimental operation: {key}"
        )

# Aggregate official test comparison.

aggregate_records = []

for metric in metric_names:

    baseline_value = float(
        verified_test_metrics["RF-B-NSL"][metric]
    )

    tuned_value = float(
        verified_test_metrics["RF-H2-NSL"][metric]
    )

    aggregate_records.append({
        "Metric": metric,
        "RF-B-NSL": baseline_value,
        "RF-H2-NSL": tuned_value,
        "Change": tuned_value - baseline_value
    })

NSL10_TEST_COMPARISON = pd.DataFrame(
    aggregate_records
)

print("\nOFFICIAL TEST COMPARISON")
print(
    NSL10_TEST_COMPARISON.to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}"
    )
)

# Per-class official test comparison.

class_records = []

for class_name in NSL10_CLASS_NAMES:

    baseline_report = verified_test_reports[
        "RF-B-NSL"
    ]

    tuned_report = verified_test_reports[
        "RF-H2-NSL"
    ]

    baseline_row = baseline_report.loc[class_name]
    tuned_row = tuned_report.loc[class_name]

    baseline_support = int(
        baseline_row["support"]
    )

    tuned_support = int(
        tuned_row["support"]
    )

    if baseline_support != tuned_support:
        raise RuntimeError(
            f"Class support mismatch: {class_name}"
        )

    class_records.append({
        "Class": class_name,
        "Support": baseline_support,
        "Baseline Precision": float(
            baseline_row["precision"]
        ),
        "Tuned Precision": float(
            tuned_row["precision"]
        ),
        "Precision Change": float(
            tuned_row["precision"]
            - baseline_row["precision"]
        ),
        "Baseline Recall": float(
            baseline_row["recall"]
        ),
        "Tuned Recall": float(
            tuned_row["recall"]
        ),
        "Recall Change": float(
            tuned_row["recall"]
            - baseline_row["recall"]
        ),
        "Baseline F1": float(
            baseline_row["f1-score"]
        ),
        "Tuned F1": float(
            tuned_row["f1-score"]
        ),
        "F1 Change": float(
            tuned_row["f1-score"]
            - baseline_row["f1-score"]
        )
    })

NSL10_TEST_CLASS_COMPARISON = pd.DataFrame(
    class_records
)

if int(
    NSL10_TEST_CLASS_COMPARISON["Support"].sum()
) != 22544:
    raise RuntimeError(
        "Class support total mismatch."
    )

print("\nPER-CLASS OFFICIAL TEST COMPARISON")
print(
    NSL10_TEST_CLASS_COMPARISON[
        [
            "Class",
            "Support",
            "Baseline F1",
            "Tuned F1",
            "F1 Change",
            "Recall Change"
        ]
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}"
    )
)

# Prediction-level agreement.

baseline_predictions = (
    verified_test_predictions["RF-B-NSL"]
)

tuned_predictions = (
    verified_test_predictions["RF-H2-NSL"]
)

baseline_correct = (
    baseline_predictions == NSL10_TEST_Y
)

tuned_correct = (
    tuned_predictions == NSL10_TEST_Y
)

a = int(np.count_nonzero(
    baseline_correct & tuned_correct
))

b = int(np.count_nonzero(
    baseline_correct & ~tuned_correct
))

c = int(np.count_nonzero(
    ~baseline_correct & tuned_correct
))

d = int(np.count_nonzero(
    ~baseline_correct & ~tuned_correct
))

prediction_disagreements = int(
    np.count_nonzero(
        baseline_predictions != tuned_predictions
    )
)

assert a + b + c + d == 22544
assert a + b == int(np.count_nonzero(baseline_correct))
assert a + c == int(np.count_nonzero(tuned_correct))
assert c - b == (
    int(np.count_nonzero(tuned_correct))
    - int(np.count_nonzero(baseline_correct))
)

validation_baseline = np.load(
    NSL10_MODEL_RESULT_DIRS["RF-B-NSL"]
    / "RF_B_NSL_validation_predictions.npy",
    allow_pickle=False
)

validation_tuned = np.load(
    NSL10_MODEL_RESULT_DIRS["RF-H2-NSL"]
    / "RF_H2_NSL_validation_predictions.npy",
    allow_pickle=False
)

validation_disagreements = int(
    np.count_nonzero(
        validation_baseline != validation_tuned
    )
)

agreement_record = {
    "dataset": "NSL-KDD",
    "baseline_model": "RF-B-NSL",
    "tuned_model": "RF-H2-NSL",
    "validation_prediction_disagreements":
        validation_disagreements,
    "official_test_prediction_disagreements":
        prediction_disagreements,
    "both_correct_a": a,
    "baseline_only_correct_b": b,
    "tuned_only_correct_c": c,
    "both_incorrect_d": d,
    "discordant_correctness_pairs": b + c,
    "net_correct_change": c - b,
    "official_test_observations": 22544
}

print("\nPREDICTION AGREEMENT")
print(
    "Validation prediction disagreements:",
    validation_disagreements
)
print(
    "Official test prediction disagreements:",
    prediction_disagreements
)
print("Both correct:", a)
print("Baseline only correct:", b)
print("Tuned only correct:", c)
print("Both incorrect:", d)
print("Net correct change:", c - b)

# Export comparison and agreement records.

comparison_exports = {
    "NSL10_official_test_model_comparison.csv":
        NSL10_TEST_COMPARISON.to_csv(
            index=False,
            float_format="%.9f"
        ).encode("utf-8"),

    "NSL10_official_test_class_comparison.csv":
        NSL10_TEST_CLASS_COMPARISON.to_csv(
            index=False,
            float_format="%.9f"
        ).encode("utf-8"),

    "NSL10_prediction_agreement.json":
        (
            json.dumps(
                agreement_record,
                indent=2,
                sort_keys=True
            ) + "\n"
        ).encode("utf-8")
}

print("\nCOMPARISON EXPORTS")

for filename, content in comparison_exports.items():

    status = ns10_test_write_once(
        NSL10_RESULTS_DIR / filename,
        content
    )

    print(f"{filename}: {status}")

# Model directory SHA-256 fingerprints.

def ns10_sha256(path):

    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:

        for chunk in iter(
            lambda: handle.read(8 * 1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

def ns10_model_fingerprint(model_path):

    model_path = Path(model_path)

    files = sorted(
        [
            path for path in model_path.rglob("*")
            if path.is_file()
        ],
        key=lambda path: str(
            path.relative_to(model_path)
        )
    )

    if not files:
        raise RuntimeError(
            f"No saved model files: {model_path}"
        )

    records = {
        str(path.relative_to(model_path)):
            ns10_sha256(path)
        for path in files
    }

    combined = hashlib.sha256(
        json.dumps(
            records,
            sort_keys=True,
            separators=(",", ":")
        ).encode("utf-8")
    ).hexdigest()

    return {
        "path": str(model_path),
        "file_count": len(files),
        "combined_sha256": combined,
        "file_sha256": records
    }

model_fingerprints = {
    model_name: ns10_model_fingerprint(
        NSL10_MODEL_PATHS[model_name]
    )
    for model_name in model_specifications
}

# Fingerprint required experimental result files.

required_result_paths = [
    locked_configuration_path,
    NSL10_RESULTS_DIR
    / "NSL10_validation_model_comparison.csv",
    NSL10_RESULTS_DIR
    / "NSL10_validation_class_comparison.csv"
]

for model_name, prefix in model_specifications.items():

    directory = NSL10_MODEL_RESULT_DIRS[model_name]

    for partition in ["validation", "test"]:

        required_result_paths.extend([
            directory / f"{prefix}_{partition}_metrics.csv",
            directory
            / f"{prefix}_{partition}_classification_report.csv",
            directory
            / f"{prefix}_{partition}_confusion_matrix.csv",
            directory
            / f"{prefix}_{partition}_predictions.npy"
        ])

required_result_paths.extend([
    NSL10_RESULTS_DIR / filename
    for filename in comparison_exports
])

if len(set(required_result_paths)) != len(
    required_result_paths
):
    raise RuntimeError(
        "Duplicate required result paths."
    )

result_fingerprints = {}

for path in required_result_paths:

    if not path.is_file():
        raise FileNotFoundError(path)

    relative = str(
        path.relative_to(PROJECT_ROOT)
    )

    result_fingerprints[relative] = ns10_sha256(path)

# Persist the final experiment manifest.

NSL10_FINAL_MANIFEST = {
    "notebook": "10_NSLKDD_Random_Forest_Replication",
    "dataset": "NSL-KDD",
    "class_order": NSL10_CLASS_NAMES,
    "encoded_predictors": 122,
    "development_training_records": 107077,
    "validation_records": 18896,
    "official_test_records": 22544,
    "model_configurations": NSL10_MODEL_CONFIGS,
    "source_preprocessing_fingerprint":
        NSL10_EXPECTED_FINGERPRINT,
    "model_fingerprints": model_fingerprints,
    "result_file_fingerprints": result_fingerprints,
    "validation_prediction_disagreements":
        validation_disagreements,
    "test_prediction_disagreements":
        prediction_disagreements,
    "validation_derived_configuration_changes": False,
    "hyperparameter_search_performed": False,
    "synthetic_resampling_performed": False,
    "cic_model_transfer_performed": False,
    "official_test_used_for_training": False,
    "official_test_used_for_model_selection": False,
    "official_test_evaluation_completed": True
}

manifest_path = (
    NSL10_RESULTS_DIR / "NSL10_manifest.json"
)

manifest_status = ns10_test_write_once(
    manifest_path,
    (
        json.dumps(
            NSL10_FINAL_MANIFEST,
            indent=2,
            sort_keys=True,
            allow_nan=False
        ) + "\n"
    ).encode("utf-8")
)

persisted_manifest = json.loads(
    manifest_path.read_text(encoding="utf-8")
)

if persisted_manifest != NSL10_FINAL_MANIFEST:
    raise RuntimeError(
        "Final manifest verification failed."
    )

for relative_path, expected_hash in (
    persisted_manifest["result_file_fingerprints"].items()
):

    if ns10_sha256(
        PROJECT_ROOT / relative_path
    ) != expected_hash:
        raise RuntimeError(
            f"Result fingerprint mismatch: {relative_path}"
        )

for model_name, recorded in (
    persisted_manifest["model_fingerprints"].items()
):

    computed = ns10_model_fingerprint(
        NSL10_MODEL_PATHS[model_name]
    )

    if computed["combined_sha256"] != (
        recorded["combined_sha256"]
    ):
        raise RuntimeError(
            f"Model fingerprint mismatch: {model_name}"
        )

print("\nFINAL AUDIT")
print("Preprocessing fingerprint: PASS")
print("Frozen model configurations: PASS")
print("Baseline test metrics and report: PASS")
print("Tuned test metrics and report: PASS")
print("Baseline confusion matrix: PASS")
print("Tuned confusion matrix: PASS")
print("Saved prediction alignment: PASS")
print("Validation and test comparisons: PASS")
print("Model directory fingerprints: PASS")
print("Result file fingerprints: PASS")
print("Persisted manifest: PASS")
print(
    "Experimental result files verified:",
    len(result_fingerprints)
)

# Notebook completion record.

completion = {
    "notebook": "10_NSLKDD_Random_Forest_Replication",
    "status": "COMPLETE",
    "dataset": "NSL-KDD",
    "trained_models": list(model_specifications),
    "validation_records": 18896,
    "official_test_records": 22544,
    "official_test_baseline_macro_f1": (
        verified_test_metrics["RF-B-NSL"]["Macro F1"]
    ),
    "official_test_tuned_macro_f1": (
        verified_test_metrics["RF-H2-NSL"]["Macro F1"]
    ),
    "official_test_macro_f1_difference": (
        verified_test_metrics["RF-H2-NSL"]["Macro F1"]
        - verified_test_metrics["RF-B-NSL"]["Macro F1"]
    ),
    "test_prediction_disagreements":
        prediction_disagreements,
    "manifest_path": str(manifest_path),
    "manifest_sha256": ns10_sha256(manifest_path),
    "audit_checks_passed": True,
    "next_stage": "11_Cross_Benchmark_Robustness_Analysis"
}

completion_path = (
    NSL10_RESULTS_DIR
    / "NSL10_NOTEBOOK_COMPLETE.json"
)

completion_status = ns10_test_write_once(
    completion_path,
    (
        json.dumps(
            completion,
            indent=2,
            sort_keys=True,
            allow_nan=False
        ) + "\n"
    ).encode("utf-8")
)

saved_completion = json.loads(
    completion_path.read_text(encoding="utf-8")
)

if saved_completion != completion:
    raise RuntimeError(
        "Notebook completion record verification failed."
    )

NSL10_NOTEBOOK_COMPLETE = True

print("\nCOMPLETION RECORD")
print("Manifest:", manifest_status)
print("Completion record:", completion_status)
print("Status:", saved_completion["status"])
print("Models:", saved_completion["trained_models"])
print(
    "Baseline official test Macro F1:",
    f"{completion['official_test_baseline_macro_f1']:.6f}"
)
print(
    "Tuned official test Macro F1:",
    f"{completion['official_test_tuned_macro_f1']:.6f}"
)
print(
    "Macro F1 difference:",
    f"{completion['official_test_macro_f1_difference']:+.6f}"
)
print(
    "Prediction disagreements:",
    completion["test_prediction_disagreements"]
)

print("\n" + "=" * 72)
print("NOTEBOOK 10: COMPLETE")
print("Models retrained during final evaluation: NO")
print("Next stage: 11_Cross_Benchmark_Robustness_Analysis")
print("=" * 72)


CELL 12 — FINAL EVALUATION AND COMPLETION AUDIT

SAVED OFFICIAL TEST RESULT VERIFICATION
RF-B-NSL: PASS
RF-H2-NSL: PASS

OFFICIAL TEST COMPARISON
           Metric  RF-B-NSL  RF-H2-NSL   Change
         Accuracy  0.758384   0.758384 0.000000
  Macro Precision  0.766484   0.766484 0.000000
     Macro Recall  0.493121   0.493121 0.000000
         Macro F1  0.504649   0.504649 0.000000
      Weighted F1  0.717539   0.717539 0.000000
Balanced Accuracy  0.493121   0.493121 0.000000
              MCC  0.650888   0.650888 0.000000

PER-CLASS OFFICIAL TEST COMPARISON
 Class  Support  Baseline F1  Tuned F1  F1 Change  Recall Change
Normal     9711     0.783829  0.783829   0.000000       0.000000
   DoS     7460     0.880332  0.880332   0.000000       0.000000
 Probe     2421     0.699855  0.699855   0.000000       0.000000
   R2L     2885     0.103675  0.103675   0.000000       0.000000
   U2R       67     0.055556  0.055556   0.000000       0.000000

PREDICTION AGREEMENT
Validation prediction 